# Peptide microarray analysis of protein–RNA binding

An end-to-end pipeline for quantifying how individual amino acids contribute to RNA binding, using peptide microarrays. Starting from raw GenePix scans it aggregates dual-channel fluorescence, deconvolves an additive per-residue contribution model by non-negative least squares, exports interpretable scales, and produces the summary figures. The sections below follow that workflow in order.

**Contents**

1. [Parse a .gal peptide library](#sec1)
2. [Peptide-chip additive-scale deconvolution](#sec2)
3. [Rebuild human-readable scales table](#sec3)
4. [Binding-scale visualisation](#sec4)
5. [Target-specific array plots](#sec5)
6. [Homopolymer binding bar plot](#sec6)

> Notebooks were consolidated from separate working scripts; unused and broken scripts were removed. Each section keeps its own imports and reads independently. Cell outputs were cleared, and input data files are referenced by generic names (not included).


<a id="sec1"></a>
## 1. Parse a .gal peptide library

Parses a GenePix Array List (.gal) file to extract the peptide identities laid out on the microarray.

<sub>Source script: <code>parse_peptide_library_gal.ipynb</code></sub>

In [ ]:
with open("ID_00097_Peptide_library.gal") as f:
    for line in f:
        print(line.strip())


In [ ]:
import re

input_file = "ID_00097_Peptide_library.gal"

peptides = []

with open(input_file, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()

        # Skip empty lines or header lines
        if not line or line.startswith('"'):
            continue

        parts = line.split()

        # Get last column
        last_col = parts[-1]

        # Check: only letters, length 10 or 11
        if re.fullmatch(r"[A-Z]{10,11}", last_col):
            peptides.append(last_col)
            print(last_col)

print("\nTotal stored peptides:", len(peptides))

with open("filtered_peptides.txt", "w") as out:
    for p in peptides:
        out.write(p + "\n")


In [ ]:
from collections import Counter

# Store matching peptides
motif_peptides = []

for pep in peptides:
    # Must be length 11
    if len(pep) != 11:
        continue

    # Check XXXGXXXGXXX pattern
    if pep[3] == "G" and pep[7] == "G":
        motif_peptides.append(pep)

# Count frequencies
motif_counts = Counter(motif_peptides)

# Results
print("Total peptides analyzed:", len(peptides))
print("Total matching XXXGXXXGXXX:", len(motif_peptides))
print("Unique matching sequences:", len(motif_counts))

print("\nTop matches (peptide -> count):")
for pep, count in motif_counts.most_common(20):
    print(f"{pep}\t{count}")


In [ ]:
import pandas as pd

# Unique matching sequences from the previous cell
# (motif_counts is a Counter; motif_peptides is a list)
unique_motifs = sorted(motif_counts.keys())

# Build a small table (also include counts, useful!)
df = pd.DataFrame({
    "peptide": unique_motifs,
    "count_in_file": [motif_counts[p] for p in unique_motifs],
})

# Write CSV (always works)
csv_path = "unique_XXXGXXXGXXX_motifs.csv"
df.to_csv(csv_path, index=False)

# Write Excel (optional but nice)
xlsx_path = "unique_XXXGXXXGXXX_motifs.xlsx"
df.to_excel(xlsx_path, index=False)

print(f"Wrote {len(df)} unique motifs to:")
print(f" - {csv_path}")
print(f" - {xlsx_path}")


In [ ]:
# Create a thesis-ready list: unique motifs only (no IDs, no counts)

# Sort alphabetically for clean presentation
unique_motifs = sorted(motif_counts.keys())

output_file = "appendix_XXXGXXXGXXX_peptides_only.txt"

with open(output_file, "w") as f:
    # Header
    f.write("Peptide Sequence\n")

    # Rows
    for peptide in unique_motifs:
        f.write(f"{peptide}\n")

print(f"List written to: {output_file}")
print(f"Total unique peptides: {len(unique_motifs)}")


In [ ]:
# Create compressed multi-column layout for Word

unique_motifs = sorted(motif_counts.keys())

N_COLS = 5   # change to 4, 5, or 6 if needed

output_file = "appendix_XXXGXXXGXXX_peptides_compact.txt"

with open(output_file, "w") as f:
    f.write("XXXGXXXGXXX Peptides\n\n")

    for i in range(0, len(unique_motifs), N_COLS):
        row = unique_motifs[i:i + N_COLS]
        f.write("\t".join(row) + "\n")

print(f"Compact list written to: {output_file}")
print(f"Columns per row: {N_COLS}")
print(f"Total peptides: {len(unique_motifs)}")


In [ ]:
import re
from collections import Counter

# Assumes you already have `peptides` (list of sequences) from your parsing cell.

# ABABABABAB where A!=B
pat_ab = re.compile(r"^([A-Z])([A-Z])(?:\1\2){4}$")
# BABABABABA where A!=B (same two letters, swapped start)
pat_ba = re.compile(r"^([A-Z])([A-Z])(?:\2\1){4}$")

ab_seqs, ba_seqs = [], []

for s in peptides:
    s = s.strip().upper()
    if len(s) != 10:
        continue

    m = pat_ab.match(s)
    if m and m.group(1) != m.group(2):
        ab_seqs.append(s)
        continue

    m = pat_ba.match(s)
    if m and m.group(1) != m.group(2):
        ba_seqs.append(s)

ab_counts = Counter(ab_seqs)
ba_counts = Counter(ba_seqs)

print(f"Total peptides (all lengths): {len(peptides)}")
print(f"Total 10-mers: {sum(1 for p in peptides if len(p)==10)}")
print(f"ABABABABAB ((AB)2) matches: {len(ab_seqs)} (unique: {len(ab_counts)})")
print(f"BABABABABA ((BA)2) matches: {len(ba_seqs)} (unique: {len(ba_counts)})")
print(f"Either alternating 10-mer: {len(ab_seqs) + len(ba_seqs)}")

# Optional: count by amino-acid pair
# For AB scheme, pair is (A,B) = (seq[0], seq[1])
ab_pairs = Counter((s[0], s[1]) for s in ab_seqs)
# For BA scheme, pair is (B,A) = (seq[0], seq[1]) as written
ba_pairs = Counter((s[0], s[1]) for s in ba_seqs)

print("\nTop (AB)2 pairs (A,B) -> count:")
for (a, b), c in ab_pairs.most_common(20):
    print(f"{a}{b}\t{c}")

print("\nTop (BA)2 pairs (B,A) -> count:")
for (b, a), c in ba_pairs.most_common(20):
    print(f"{b}{a}\t{c}")


In [ ]:
import re
from pathlib import Path

AA10 = re.compile(r"^[A-Z]{10}$")

def extract_10mers_from_gal(path: str) -> list[str]:
    """Extract 10-mer sequences from a .gal file; assumes peptide is last whitespace token."""
    seqs = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('"'):
                continue
            parts = line.split()
            if not parts:
                continue
            s = parts[-1].strip().upper()
            if AA10.fullmatch(s):
                seqs.append(s)
    return seqs

# ---- choose inputs ----
# Single file:
gal_files = ["ID_00097_Peptide_library.gal"]

# Or if you have multiple library files in a folder, use:
# gal_files = [str(p) for p in Path(".").glob("*.gal")]

all_10mers = []
for fp in gal_files:
    all_10mers.extend(extract_10mers_from_gal(fp))

print("Total 10-mers parsed (with duplicates):", len(all_10mers))
uniq_10mers = sorted(set(all_10mers))
print("Unique 10-mers:", len(uniq_10mers))


In [ ]:
from collections import Counter
import pandas as pd

# Use unique motif peptides only
unique_motifs = set(sorted(motif_counts.keys()))

aa_counts = Counter()
total = 0

for seq in unique_motifs:
    for aa in seq:
        aa_counts[aa] += 1
        total += 1

# Build table
df = pd.DataFrame(
    [
        {
            "Amino_acid": aa,
            "Count": cnt,
            "Percent": round(cnt / total * 100, 2)
        }
        for aa, cnt in aa_counts.items()
    ]
).sort_values("Count", ascending=False)

print("Unique XXXGXXXGXXX peptides:", len(unique_motifs))
print("Total residues analyzed:", total)
print()
print(df.to_string(index=False))

# Optional: save for thesis
df.to_csv("XXXGXXXGXXX_aa_composition.csv", index=False)


<a id="sec2"></a>
## 2. Peptide-chip additive-scale deconvolution

Reads a raw peptide-microarray scan (GenePix export), aggregates red/green fluorescence per peptide and block, then fits a non-negative least-squares additive model to extract per-residue contribution scales for each RNA condition.

<sub>Source script: <code>peptide_chip_scale_deconvolution.ipynb</code></sub>

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

# same dictionaries used in your script
red_I = defaultdict(lambda: defaultdict(list))
green_I = defaultdict(lambda: defaultdict(list))

path = "chip_scan.txt"

with open(path, "rb") as file:
    for i, l in enumerate(file):

        try:
            l = l.decode("ascii")
        except:
            continue

        l = l.strip().split("\t")

        if i == 27:
            header = l
            block = header.index('"Block"')
            ID = header.index('"ID"')
            F635 = header.index('"F635 Median"')
            F532 = header.index('"F532 Median"')

        elif i > 27:

            block_id = int(l[block])
            peptide = l[ID]

            red_I[block_id][peptide].append(float(l[F635]))
            green_I[block_id][peptide].append(float(l[F532]))

# block names from your script
column_names = [nb + C for C in ['_low','_high'] for nb in 'GACU']

os.makedirs("chip_xlsx", exist_ok=True)

for block in red_I.keys():

    rows = []

    for peptide in red_I[block].keys():

        rows.append({
            "peptide": peptide,
            "F635_mean": np.mean(red_I[block][peptide]),
            "F635_std": np.std(red_I[block][peptide]),
            "F532_mean": np.mean(green_I[block][peptide]),
            "F532_std": np.std(green_I[block][peptide]),
            "N_replicates": len(red_I[block][peptide])
        })

    df = pd.DataFrame(rows)

    name = column_names[block-1]
    outfile = f"chip_xlsx/block_{block}_{name}.xlsx"

    df.to_excel(outfile, index=False)

    print("written:", outfile)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import lsq_linear
from scipy import stats
from collections import defaultdict

AAs = ['C', 'L', 'F', 'W', 'I', 'M', 'P', 'V', 'T', 'A', 'S', 'Y', 'H', 'R', 'Q', 'G', 'N', 'K', 'D', 'E']
homos = set([aa * 10 for aa in AAs])
BIs = set([(a + b) * 5 for a in AAs for b in AAs])
column_names = [nb + C for C in ['_low', '_high'] for nb in 'GACU']  # G_low ... U_high

red_I = defaultdict(lambda: defaultdict(list))
path = "chip_scan.txt"

with open(path, "rb") as file:
    for i, l in enumerate(file):
        try:
            l = l.decode("ascii")
        except:
            continue
        l = l.strip().split("\t")

        if i == 27:
            header = l
            block = header.index('"Block"')
            ID = header.index('"ID"')
            F635 = header.index('"F635 Median"')

        elif i > 27:
            peptide = l[ID]
            if peptide in homos or peptide in BIs:
                red_I[int(l[block])][peptide].append(float(l[F635]))

def encode_20(seq):
    return np.array([seq.count(aa) for aa in 'ARNDCQEGHILKMFPSTWYV'])

fig, axes = plt.subplots(2, 4, figsize=(12, 6), dpi=300)
axes = axes.ravel()

for i in range(8):
    block = i + 1
    ax = axes[i]

    X, y = [], []
    for peptide in BIs - homos:
        if peptide in red_I[block]:
            X.append(encode_20(peptide))
            y.append(np.mean(red_I[block][peptide]))

    X = np.array(X)
    y = np.array(y)

    res = lsq_linear(X, y, bounds=(-10000, 10000), method='bvls')
    expected = X @ res.x
    observed = y

    slope, intercept, r_value, p_value, std_err = stats.linregress(expected, observed)
    xx = np.array([expected.min(), expected.max()])

    ax.scatter(expected, observed, c='red', alpha=0.2, s=12)
    ax.plot(xx, intercept + slope * xx, '--', color='black', linewidth=1)

    ax.set_title(f"{column_names[i]}")
    ax.set_xlabel("compositional expected intensity")
    ax.set_ylabel("observed intensity")
    ax.text(
        0.95, 0.05,
        f"N={len(observed)}\n$R^2$={r_value**2:.3f}",
        transform=ax.transAxes,
        fontsize=8,
        va='bottom',
        ha='right'
    )

plt.tight_layout()
plt.savefig("consistency_AB5_all_panels.png")
plt.show()

In [ ]:
scale = 1000

fig, axes = plt.subplots(2, 4, figsize=(12, 6), dpi=300)
axes = axes.ravel()

for i in range(8):
    block = i + 1
    ax = axes[i]

    X, y = [], []
    for peptide in BIs - homos:
        if peptide in red_I[block]:
            X.append(encode_20(peptide))
            y.append(np.mean(red_I[block][peptide]))

    X = np.array(X)
    y = np.array(y)

    res = lsq_linear(X, y, bounds=(-10000, 10000), method='bvls')
    expected = X @ res.x
    observed = y

    slope, intercept, r_value, p_value, std_err = stats.linregress(expected, observed)

    # scale only for plotting
    expected_plot = expected / scale
    observed_plot = observed / scale
    xx = np.array([expected_plot.min(), expected_plot.max()])

    ax.scatter(expected_plot, observed_plot, c='red', alpha=0.2, s=12)
    ax.plot(xx, (intercept/scale) + slope * xx, '--', color='black', linewidth=1)

    ax.set_title(f"{column_names[i]}")
    ax.set_xlabel("expected intensity (×10³)")
    ax.set_ylabel("observed intensity (×10³)")
    ax.text(-0.20, 1.10, chr(65+i), transform=ax.transAxes, fontsize=16, fontweight='bold', va='top')
    ax.text(
        0.95, 0.05,
        f"N={len(observed)}\n$R^2$={r_value**2:.3f}",
        transform=ax.transAxes,
        fontsize=8,
        va='bottom',
        ha='right'
    )

plt.tight_layout()
plt.savefig("consistency_AB5_all_panels_scaled.png", dpi=300)
plt.show()

In [ ]:
scale = 1000

fig, axes = plt.subplots(4, 2, figsize=(6, 12), dpi=300)
axes = axes.ravel()
order = [0,4,1,5,2,6,3,7]

for j, i in enumerate(order):
    block = i + 1
    ax = axes[j]

    # panel label
    ax.text(-0.18, 1.12, chr(65+j), transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='top')

    X, y = [], []
    for peptide in BIs - homos:
        if peptide in red_I[block]:
            X.append(encode_20(peptide))
            y.append(np.mean(red_I[block][peptide]))

    X = np.array(X)
    y = np.array(y)

    res = lsq_linear(X, y, bounds=(-10000, 10000), method='bvls')
    expected = X @ res.x
    observed = y

    slope, intercept, r_value, p_value, std_err = stats.linregress(expected, observed)

    # scale for plotting
    expected_plot = expected / scale
    observed_plot = observed / scale

    xx = np.array([expected_plot.min(), expected_plot.max()])

    ax.scatter(expected_plot, observed_plot, c='red', alpha=0.2, s=12)
    ax.plot(xx, (intercept/scale) + slope * xx, '--', color='black', linewidth=1)

    ax.set_title(column_names[i])

    ax.set_xlabel("expected intensity (×10³)")
    ax.set_ylabel("observed intensity (×10³)")

    ax.text(
        0.95, 0.05,
        f"N={len(observed)}\n$R^2$={r_value**2:.3f}",
        transform=ax.transAxes,
        fontsize=8,
        va='bottom',
        ha='right'
    )

plt.tight_layout()
plt.savefig("consistency_AB5_panels_2x4.png", dpi=300)
plt.show()

In [ ]:
scale = 1000

fig, axes = plt.subplots(2, 2, figsize=(8, 8), dpi=300)
axes = axes.ravel()

# nucleotide -> (low block, high block)
pairs = [
    ("G", 1, 5),
    ("A", 2, 6),
    ("C", 3, 7),
    ("U", 4, 8),
]

colors = {"low": "red", "high": "blue"}

for j, (nt, low_block, high_block) in enumerate(pairs):
    ax = axes[j]

    ax.text(-0.15, 1.08, chr(65 + j), transform=ax.transAxes,
            fontsize=13, fontweight='bold', va='top')

    results = {}

    for label, block in [("low", low_block), ("high", high_block)]:
        X, y = [], []
        for peptide in BIs - homos:
            if peptide in red_I[block]:
                X.append(encode_20(peptide))
                y.append(np.mean(red_I[block][peptide]))

        X = np.array(X)
        y = np.array(y)

        res = lsq_linear(X, y, bounds=(-10000, 10000), method='bvls')
        expected = X @ res.x
        observed = y

        slope, intercept, r_value, p_value, std_err = stats.linregress(expected, observed)

        results[label] = {
            "expected": expected / scale,
            "observed": observed / scale,
            "slope": slope,
            "intercept": intercept / scale,
            "r2": r_value**2,
            "n": len(observed),
        }

    for label in ["low", "high"]:
        x = results[label]["expected"]
        y = results[label]["observed"]
        slope = results[label]["slope"]
        intercept = results[label]["intercept"]

        xx = np.array([x.min(), x.max()])

        ax.scatter(x, y, alpha=0.22, s=12, color=colors[label], label=label)
        ax.plot(xx, intercept + slope * xx, "--", color=colors[label], linewidth=1)

    ax.set_title(nt)
    ax.set_xlabel("expected intensity (×10³)")
    ax.set_ylabel("observed intensity (×10³)")

    ax.text(
        0.98, 0.02,
        f"low:  $R^2$={results['low']['r2']:.3f}\n"
        f"high: $R^2$={results['high']['r2']:.3f}",
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="right"
    )

    ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("consistency_AB5_overlay_4panels.png", dpi=300)
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# =========================
# INPUTS
# =========================
fus_protein = "MASNDYTQQATQSYGAYPTQPGQGYSQQSSQPYGQQSYSGYSQSTDTSGYGQSSYSSYGQSQNTGYGTQSTPQGYGSTGGYGSSQSSQSSYGQQSSYPGYGQQPAPSSTSGSYGSSSQSSSYGQPQSGSYSQQPSYGGQQQSYGQQQSYNPPQGYGQQNQYNSSSGGGGGGGGGGNYGQDQSSMSSGGGSGGGYGNQDQSGGGGSGGYGQQDRGGRGRGGSGGGGGGGGGGYNRSSGGYEPRGRGGGRGGRGGMGGSDRGGFNKFGGPRDQGSRHDSEQDNSDNNTIFVQGLGENVTIESVADYFKQIGIIKTNKKTGQPMINLYTDRETGKLKGEATVSFDDPPSAKAAIDWFDGKEFSGNPIKVSFATRRADFNRGGGNGRGGRGRGGPMGRGGYGGGGSGGGGRGGFPSGGGGGGGQQRAGDWKCPNPTCENMNFSWRNECNQCKAPKPDGPGGGPGGSHMGGNYGDDRRGGRGGYDRGGYRGRGGDRGGFRGGRGGGDRGGFGPGKMDSRGEHRQDRRERPY"
chip_scales_path = "chip_scales.data"

# =========================
# SAME HELPER LOGIC AS SCRIPT
# =========================
def moving_average(a, n=3):
    ret = np.nancumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

def pep10(seq):
    if len(seq) == 10:
        return seq
    if len(seq) == 2:
        return seq * 5
    if len(seq) == 1:
        return seq * 10
    if len(seq) == 3:
        return seq + 'G' + seq + 'G' + seq

with open(chip_scales_path, "rb") as f:
    chip_scales = pickle.load(f)

window = 21

fig, axes = plt.subplots(2, 2, figsize=(8, 6), dpi=300)
axes = axes.ravel()

for i, NB in enumerate("GACU"):
    ax = axes[i]

    for L in [1, 2, 3, 10]:
        affinity_vec = [
            chip_scales[NB][pep10(fus_protein[j:j+L])]
            for j in range(len(fus_protein) - L + 1)
        ]

        affinity_vec = moving_average(affinity_vec, n=window)
        mean_pr = np.mean(affinity_vec)
        std_pr = np.std(affinity_vec)
        affinity_vec = (np.array(affinity_vec) - mean_pr) / std_pr

        x = np.arange(len(fus_protein))
        ax.plot(
            x[(L // 2) + (window // 2): len(affinity_vec) + (L // 2) + (window // 2)],
            affinity_vec,
            linewidth=0.7,
            linestyle=(0, (1, 0.4)),
            label=f"{L}-mer"
        )


    ax.set_title(NB)
    ax.set_xlabel("FUS sequence")
    ax.set_ylabel(f"{NB}-intensity (SD)")
    ax.text(-0.12, 1.05, chr(65 + i), transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="top")

fig.legend(handles, labels,
           loc="lower center",
           ncol=4,
           frameon=False,
           fontsize=14,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig("length_scale_comparison_FUS_combined.png", dpi=300)
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# =========================
# INPUTS
# =========================
fus_protein = "MASNDYTQQATQSYGAYPTQPGQGYSQQSSQPYGQQSYSGYSQSTDTSGYGQSSYSSYGQSQNTGYGTQSTPQGYGSTGGYGSSQSSQSSYGQQSSYPGYGQQPAPSSTSGSYGSSSQSSSYGQPQSGSYSQQPSYGGQQQSYGQQQSYNPPQGYGQQNQYNSSSGGGGGGGGGGNYGQDQSSMSSGGGSGGGYGNQDQSGGGGSGGYGQQDRGGRGRGGSGGGGGGGGGGYNRSSGGYEPRGRGGGRGGRGGMGGSDRGGFNKFGGPRDQGSRHDSEQDNSDNNTIFVQGLGENVTIESVADYFKQIGIIKTNKKTGQPMINLYTDRETGKLKGEATVSFDDPPSAKAAIDWFDGKEFSGNPIKVSFATRRADFNRGGGNGRGGRGRGGPMGRGGYGGGGSGGGGRGGFPSGGGGGGGQQRAGDWKCPNPTCENMNFSWRNECNQCKAPKPDGPGGGPGGSHMGGNYGDDRRGGRGGYDRGGYRGRGGDRGGFRGGRGGGDRGGFGPGKMDSRGEHRQDRRERPY"
fus_cdna = """ATGGCCTCAAACGATTATACCCAACAAGCAACCCAAAGCTATGGGGCCTACCCCACCCAGCCCGGGCAGGGCTATTCCCAGCAGAGCAGTCAGCCCTACGGACAGCAGAGTTACAGTGGTTATAGCCAGTCCACGGACACTTCAGGCTATGGCCAGAGCAGCTATTCTTCTTATGGCCAGAGCCAGAACACAGGCTATGGAACTCAGTCAACTCCCCAGGGATATGGCTCGACTGGCGGCTATGGCAGTAGCCAGAGCTCCCAATCGTCTTACGGGCAGCAGTCCTCCTACCCTGGCTATGGCCAGCAGCCAGCTCCCAGCAGCACCTCGGGAAGTTACGGTAGCAGTTCTCAGAGCAGCAGCTATGGGCAGCCCCAGAGTGGGAGCTACAGCCAGCAGCCTAGCTATGGTGGACAGCAGCAAAGCTATGGACAGCAGCAAAGCTATAATCCCCCTCAGGGCTATGGACAGCAGAACCAGTACAACAGCAGCAGTGGTGGTGGAGGTGGAGGTGGAGGTGGAGGTAACTATGGCCAAGATCAATCCTCCATGAGTAGTGGTGGTGGCAGTGGTGGCGGTTATGGCAATCAAGACCAGAGTGGTGGAGGTGGCAGCGGTGGCTATGGACAGCAGGACCGTGGAGGCCGCGGCAGGGGTGGCAGTGGTGGCGGCGGCGGCGGCGGCGGTGGTGGTTACAACCGCAGCAGTGGTGGCTATGAACCCAGAGGTCGTGGAGGTGGCCGTGGAGGCAGAGGTGGCATGGGCGGAAGTGACCGTGGTGGCTTCAATAAATTTGGTGGCCCTCGGGACCAAGGATCACGTCATGACTCCGAACAGGATAATTCAGACAACAACACCATCTTTGTGCAAGGCCTGGGTGAGAATGTTACAATTGAGTCTGTGGCTGATTACTTCAAGCAGATTGGTATTATTAAGACAAACAAGAAAACGGGACAGCCCATGATTAATTTGTACACAGACAGGGAAACTGGCAAGCTGAAGGGAGAGGCAACGGTCTCTTTTGATGACCCACCTTCAGCTAAAGCAGCTATTGACTGGTTTGATGGTAAAGAATTCTCCGGAAATCCTATCAAGGTCTCATTTGCTACTCGCCGGGCAGACTTTAATCGGGGTGGTGGCAATGGTCGTGGAGGCCGAGGGCGAGGAGGACCCATGGGCCGTGGAGGCTATGGAGGTGGTGGCAGTGGTGGTGGTGGCCGAGGAGGATTTCCCAGTGGAGGTGGTGGCGGTGGAGGACAGCAGCGAGCTGGTGACTGGAAGTGTCCTAATCCCACCTGTGAGAATATGAACTTCTCTTGGAGGAATGAATGCAACCAGTGTAAGGCCCCTAAACCAGATGGCCCAGGAGGGGGACCAGGTGGCTCTCACATGGGGGGTAACTACGGGGATGATCGTCGTGGTGGCAGAGGAGGCTATGATCGAGGCGGCTACCGGGGCCGCGGCGGGGACCGTGGAGGCTTCCGAGGGGGCCGGGGTGGTGGGGACAGAGGTGGCTTTGGCCCTGGCAAGATGGATTCCAGGGGTGAGCACAGACAGGATCGCAGGGAGAGGCCGTATTAA"""
fus_utr5_length = 0

chip_scales_path = "chip_scales.data"

NB = "G"            # choose: G, A, C, U
against_PUR = True  # True gives PUR-content; False gives NB-content

# =========================
# SAME HELPER LOGIC AS SCRIPT
# =========================
def moving_average(a, n=3):
    ret = np.nancumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

def pep10(seq):
    if len(seq) == 10:
        return seq
    if len(seq) == 2:
        return seq * 5
    if len(seq) == 1:
        return seq * 10
    if len(seq) == 3:
        return seq + 'G' + seq + 'G' + seq

with open(chip_scales_path, "rb") as f:
    chip_scales = pickle.load(f)

window = 21 * 3
start_i = fus_utr5_length

fig, axes = plt.subplots(4, 1, figsize=(5, 14), dpi=300)

for i, L in enumerate([1, 2, 3, 10]):
    ax = axes[i]

    # RNA track: same logic as script
    if against_PUR:
        cont = [1 if nb in "GA" else 0 for nb in fus_cdna]
    else:
        cont = [1 if nb == NB else 0 for nb in fus_cdna]

    cont = moving_average(cont, n=window)
    mean_cont = np.mean(cont[start_i:start_i + len(fus_protein) * 3])
    std_cont = np.std(cont[start_i:start_i + len(fus_protein) * 3])
    cont = (np.array(cont) - mean_cont) / std_cont

    # protein affinity track: same logic as script
    affinity_vec = [
        chip_scales[NB][pep10(fus_protein[j:j+L])]
        for j in range(len(fus_protein) - L + 1)
        for _ in range(3)
    ]
    affinity_vec = moving_average(affinity_vec, n=window)
    mean_aff = np.mean(affinity_vec)
    std_aff = np.std(affinity_vec)
    affinity_vec = (np.array(affinity_vec) - mean_aff) / std_aff

    # same correlation logic
    x1 = cont[start_i + (L // 2): start_i + len(affinity_vec) + (L // 2)]
    x2 = affinity_vec
    R = stats.pearsonr(x1, x2)[0]

    x = np.arange(len(fus_cdna))

    ax.plot(
        x[(window // 2): len(cont) + (window // 2)],
        cont,
        linewidth=0.7,
        color="black",
        alpha=0.6,
        label="mRNA"
    )

    ax.plot(
        x[start_i + (L // 2) + (window // 2): start_i + len(affinity_vec) + (L // 2) + (window // 2)],
        affinity_vec,
        color="#d95f5f",
        alpha=0.7,
        linewidth=0.7,
        linestyle="--",
        label="protein"
    )

    ax.text(
        0.12, 0.95,
        f"FUS, R ={np.round(R, 2)} (CDS)",
        va="top", ha="left",
        transform=ax.transAxes,
        fontsize=9
    )

    ax.set_xlabel("nt | AA×3")
    ax.set_ylabel("PUR-content" if against_PUR else f"{NB}-content")

    ax_twin = ax.twinx()
    lims = ax.get_ylim()
    ax_twin.set_ylim([lims[0], lims[1]])
    ax_twin.set_ylabel(f"{NB}-affinity ({L}-mere)", rotation=-90, labelpad=10, color="#d95f5f")
    ax_twin.tick_params(axis="y", colors="#d95f5f")

    ax.legend(frameon=False, loc="upper right", fontsize=8)
    ax.text(-0.08, 1.03, "ACEG"[i], transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top")

    ax.grid(False)
    ax_twin.grid(False)

plt.tight_layout()
plt.savefig(f"profile_FUS_{'PUR' if against_PUR else NB}_vertical.png", dpi=300)
plt.show()

In [ ]:
window = 21 * 3
start_i = fus_utr5_length

fig, axes = plt.subplots(2, 2, figsize=(8, 6), dpi=300)
axes = axes.ravel()

for i, L in enumerate([1,2,3,10]):

    ax = axes[i]

    # RNA content
    if against_PUR:
        cont = [1 if nb in "GA" else 0 for nb in fus_cdna]
    else:
        cont = [1 if nb == NB else 0 for nb in fus_cdna]

    cont = moving_average(cont, n=window)

    mean_cont = np.mean(cont[start_i:start_i+len(fus_protein)*3])
    std_cont = np.std(cont[start_i:start_i+len(fus_protein)*3])
    cont = (np.array(cont) - mean_cont) / std_cont

    # protein affinity
    affinity_vec = [
        chip_scales[NB][pep10(fus_protein[j:j+L])]
        for j in range(len(fus_protein)-L+1)
        for _ in range(3)
    ]

    affinity_vec = moving_average(affinity_vec, n=window)

    mean_aff = np.mean(affinity_vec)
    std_aff = np.std(affinity_vec)
    affinity_vec = (np.array(affinity_vec) - mean_aff) / std_aff

    # correlation
    R = stats.pearsonr(
        cont[start_i+(L//2):start_i+len(affinity_vec)+(L//2)],
        affinity_vec
    )[0]

    x = np.arange(len(fus_cdna))

    ax.plot(
        x[(window//2):len(cont)+(window//2)],
        cont,
        color="black",
        linewidth=0.7,
        alpha=0.6,
        label="mRNA"
    )

    ax.plot(
        x[start_i+(L//2)+(window//2): start_i+len(affinity_vec)+(L//2)+(window//2)],
        affinity_vec,
        color="red",
        linestyle="--",
        linewidth=0.7,
        alpha=0.7,
        label="protein"
    )

    ax.text(
        0.05,0.95,
        f"R={np.round(R,2)}",
        transform=ax.transAxes,
        va="top"
    )

    ax.set_xlabel("nt | AA×3")
    ax.set_ylabel("PUR-content" if against_PUR else f"{NB}-content")

    ax_twin = ax.twinx()
    ax_twin.set_ylim(ax.get_ylim())
    ax_twin.set_ylabel(f"{NB}-affinity ({L}-mer)", color="red")
    ax_twin.tick_params(axis='y', colors="red")

    ax.legend(frameon=False, fontsize=8, loc="upper right")

    # panel labels
    ax.text(-0.12,1.05,"ABCD"[i],transform=ax.transAxes,
            fontsize=13,fontweight="bold")

plt.tight_layout()
plt.savefig("FUS_profile_2x2.png",dpi=300)
plt.show()

In [ ]:
def draw_fus_domains(ax, y=-0.18):
    domains = [
        ("PrLD", 1, 239),
        ("RGG1", 285, 371),
        ("RRM", 371, 454),
        ("RGG2", 454, 501),
        ("ZnF", 422, 453),
        ("RGG3", 501, 526),
    ]

    for name, start, end in domains:
        ax.plot([start*3, end*3], [y, y], transform=ax.get_xaxis_transform(),
                color="black", lw=1, clip_on=False)
        ax.text(((start+end)/2)*3, y-0.03, name,
                transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=8)

    ax.plot([1*3, 526*3], [y, y], transform=ax.get_xaxis_transform(),
            color="black", lw=1, clip_on=False)
    ax.text(1*3, y-0.03, "1", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8)
    ax.text(526*3, y-0.03, "526", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8)

In [ ]:
window = 21 * 3
start_i = fus_utr5_length

fig, axes = plt.subplots(2, 2, figsize=(8, 6), dpi=300)
axes = axes.ravel()

for i, L in enumerate([1,2,3,10]):

    ax = axes[i]

    # RNA content
    if against_PUR:
        cont = [1 if nb in "GA" else 0 for nb in fus_cdna]
    else:
        cont = [1 if nb == NB else 0 for nb in fus_cdna]

    cont = moving_average(cont, n=window)

    mean_cont = np.mean(cont[start_i:start_i+len(fus_protein)*3])
    std_cont = np.std(cont[start_i:start_i+len(fus_protein)*3])
    cont = (np.array(cont) - mean_cont) / std_cont

    # protein affinity
    affinity_vec = [
        chip_scales[NB][pep10(fus_protein[j:j+L])]
        for j in range(len(fus_protein)-L+1)
        for _ in range(3)
    ]

    affinity_vec = moving_average(affinity_vec, n=window)

    mean_aff = np.mean(affinity_vec)
    std_aff = np.std(affinity_vec)
    affinity_vec = (np.array(affinity_vec) - mean_aff) / std_aff

    # correlation
    R = stats.pearsonr(
        cont[start_i+(L//2):start_i+len(affinity_vec)+(L//2)],
        affinity_vec
    )[0]

    x = np.arange(len(fus_cdna))

    ax.plot(
        x[(window//2):len(cont)+(window//2)],
        cont,
        color="black",
        linewidth=0.7,
        alpha=0.6,
        label="mRNA"
    )

    ax.plot(
        x[start_i+(L//2)+(window//2): start_i+len(affinity_vec)+(L//2)+(window//2)],
        affinity_vec,
        color="red",
        linestyle="--",
        linewidth=0.7,
        alpha=0.7,
        label="protein"
    )

    ax.text(
        0.05,0.95,
        f"R={np.round(R,2)}",
        transform=ax.transAxes,
        va="top"
    )

    ax.set_xlabel("nt | AA×3")
    ax.set_ylabel("PUR-content" if against_PUR else f"{NB}-content")

    ax_twin = ax.twinx()
    ax_twin.set_ylim(ax.get_ylim())
    ax_twin.set_ylabel(f"{NB}-affinity ({L}-mer)", color="red")
    ax_twin.tick_params(axis='y', colors="red")

    ax.legend(frameon=False, fontsize=8, loc="upper right")

    draw_fus_domains(ax)    # panel labels
    ax.text(-0.12,1.05,"ABCD"[i],transform=ax.transAxes,
            fontsize=13,fontweight="bold")

plt.tight_layout()
plt.savefig("FUS_profile_2x2.png",dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# same block naming as in your script
column_names = [nb + C for C in ['_low', '_high'] for nb in 'GACU']

# build di_values exactly like in the original script
di_values = []
for BLOCK in red_I.keys():
    di_val = []
    for bi in BIs:
        di_val.append(np.mean(red_I[BLOCK][bi]))
    di_values.append(di_val)

# nucleotide pairs: low block, high block
pairs = [
    ("G", 0, 4),
    ("A", 1, 5),
    ("C", 2, 6),
    ("U", 3, 7),
]

fig, axes = plt.subplots(2, 2, figsize=(8, 8), dpi=300)
axes = axes.ravel()

for i, (nt, low_idx, high_idx) in enumerate(pairs):
    ax = axes[i]

    # panel label
    ax.text(-0.10, 1.06, "ABCD"[i], transform=ax.transAxes,
            fontsize=16, fontweight="bold", va="top", ha="right")

    scale = 1000
    x = np.array(di_values[low_idx]) / scale
    y = np.array(di_values[high_idx]) / scale

    ax.scatter(x, y, s=8, color='grey', alpha=0.5, zorder=0)

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    xx = np.array([x.min(), x.max()])
    ax.plot(xx, intercept + slope * xx, '--', color='black', linewidth=1)

    ax.set_xlabel(f"{nt} low intensity")
    ax.set_ylabel(f"{nt} high intensity")

    ax.text(
        0.95, 0.05,
        f"{len(y)} peptides\nR={r_value:.2f}",
        transform=ax.transAxes,
        fontsize=8,
        va='bottom',
        ha='right'
    )

plt.tight_layout()
plt.savefig("high_vs_low_log_comparison_ABCD.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def add_dict_branch(d, keys, value):
    cur = d
    for k in keys[:-1]:
        if k not in cur:
            cur[k] = {}
        cur = cur[k]
    if keys[-1] not in cur:
        cur[keys[-1]] = value

def abline(slope, intercept, color='blue', ax='standard', alpha=0.5):
    if ax == 'standard':
        ax = plt.gca()
    x_vals = np.array(ax.get_xlim())
    y_vals = intercept + slope * x_vals
    ax.plot(x_vals, y_vals, '--', color='black', alpha=alpha, linewidth=1)

def fig_n_grid(width=9, grid_spec=[4, 2], aspect_ratio=2/3,
               left=0.09, right=0.97, bottom=0.05, top=0.97,
               wspace=0.25, hspace='standard', dpi=300):
    if hspace == 'standard':
        hspace = (wspace / aspect_ratio**(3/4))

    panel_w = width * (right-left) * (1-((grid_spec[1]-1)/grid_spec[1])*wspace) / grid_spec[1]
    height = panel_w * aspect_ratio / (1-((grid_spec[0]-1)/grid_spec[0])*hspace) * grid_spec[0] / (top-bottom)

    fig = plt.figure(figsize=[width, height], dpi=dpi)
    grid = fig.add_gridspec(grid_spec[0], grid_spec[1],
                            left=left, right=right, bottom=bottom, top=top,
                            wspace=wspace, hspace=hspace)
    return fig, grid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# raw file
old_path = "chip_scan.txt"

# small helper matching the original nested-dict usage
def add_dict_branch(d, keys, value):
    cur = d
    for k in keys[:-1]:
        if k not in cur:
            cur[k] = {}
        cur = cur[k]
    if keys[-1] not in cur:
        cur[keys[-1]] = value

def abline(slope, intercept, color='blue', ax='standard', alpha=0.5):
    if ax == 'standard':
        ax = plt.gca()
    x_vals = np.array(ax.get_xlim())
    y_vals = intercept + slope * x_vals
    ax.plot(x_vals, y_vals, '--', color='black', alpha=alpha, linewidth=1)

def fig_n_grid(width=9, grid_spec=[4, 2], aspect_ratio=2/3,
               left=0.09, right=0.97, bottom=0.05, top=0.97,
               wspace=0.25, hspace='standard', dpi=300):
    if hspace == 'standard':
        hspace = (wspace / aspect_ratio**(3/4))
    panel_w = width * (right-left) * (1-((grid_spec[1]-1)/grid_spec[1])*wspace) / grid_spec[1]
    height = panel_w * aspect_ratio / (1-((grid_spec[0]-1)/grid_spec[0])*hspace) * grid_spec[0] / (top-bottom)
    fig = plt.figure(figsize=[width, height], dpi=dpi)
    grid = fig.add_gridspec(grid_spec[0], grid_spec[1],
                            left=left, right=right, bottom=bottom, top=top,
                            wspace=wspace, hspace=hspace)
    return fig, grid

# -----------------------------
# original adjacency-data build
# -----------------------------
min_pos, max_pos = 1, 78

pos_to_F635 = {}
pos_to_ID = {}
ID_to_pos = {}
ID_to_F635 = {}

with open(old_path, 'rb') as file:
    for i, l in enumerate(file):
        try:
            l = l.decode('ascii')
        except:
            continue

        l = l.strip()
        l = l.split('\t')

        if i == 27:
            header = l
            block = header.index('"Block"')
            ID = header.index('"ID"')
            F635 = header.index('"F635 Median"')
            F532 = header.index('"F532 Median"')
            column = header.index('"Column"')
            row = header.index('"Row"')

        elif i > 27:
            add_dict_branch(pos_to_F635, [int(l[block]), int(l[row]), int(l[column])], float(l[F635]))
            add_dict_branch(pos_to_ID, [int(l[block]), int(l[row]), int(l[column])], l[ID])
            add_dict_branch(ID_to_pos, [int(l[block]), l[ID]], [])
            ID_to_pos[int(l[block])][l[ID]].append([int(l[row]), int(l[column])])
            add_dict_branch(ID_to_F635, [int(l[block]), l[ID]], [])
            ID_to_F635[int(l[block])][l[ID]].append(float(l[F635]))

# bottom 10% of IDs
lowest_per_block = {}
for block in range(1, 9):
    Is = []
    for ID in ID_to_F635[block].keys():
        I = np.mean(ID_to_F635[block][ID])
        Is.append([I, ID])
    Is.sort()
    lowest_per_block[block] = [Is[i][1] for i in range(len(Is)//10)]

def replicates_neighbors(block, ID, mean_max='mean'):
    def surround(block, row, col):
        vals = []
        for r in [row-1, row, row+1]:
            for c in [col-1, col, col+1]:
                if not (r == row and c == col):
                    vals.append(pos_to_F635[block][r][c])
        return vals

    values, neighbors, POSs = [], [], []
    for row, col in ID_to_pos[block][ID]:
        if 1 < col < 78 and 1 < row < 78:
            values.append(pos_to_F635[block][row][col])
            neighbors.append(surround(block, row, col))
            POSs.append([row, col])

    if mean_max == 'mean':
        neighbors = [np.mean(n) for n in neighbors]
        neighbors = [n/np.mean(neighbors) for n in neighbors]
    if mean_max == 'max':
        neighbors = [max(n) for n in neighbors]
        neighbors = [n/np.mean(neighbors) for n in neighbors]
    values = [v/np.mean(values) for v in values]

    return values, neighbors, POSs

# -----------------------------
# original plotting block
# -----------------------------
mean_max = 'mean'
approach = 'all'   # change to 'all' for the other original variant

fig, grid = fig_n_grid(width=9, grid_spec=[4, 2], aspect_ratio=2/3, dpi=300)

ax1 = plt.subplot(grid[0, 0]); ax2 = plt.subplot(grid[0, 1])
ax3 = plt.subplot(grid[1, 0]); ax4 = plt.subplot(grid[1, 1])
ax5 = plt.subplot(grid[2, 0]); ax6 = plt.subplot(grid[2, 1])
ax7 = plt.subplot(grid[3, 0]); ax8 = plt.subplot(grid[3, 1])

block_names = [nb + C for C in ['_low', '_high'] for nb in 'GACU']

old_adj = {}

for i in range(8):
    block = i + 1
    ax = eval('ax' + str(i+1))

    ax.set_xlabel('F635 / replicate-mean')
    ax.set_ylabel(mean_max + ' adjacent F635\n(rel. adjacents of replicates)')

    x, y, POS = [], [], []

    if approach == 'all':
        IDs = lowest_per_block[block]
    if approach == 'EMPTY':
        IDs = ['EMPTY']

    for ID in IDs:
        rep_vals, neighbour_vals, pos = replicates_neighbors(block, ID, mean_max=mean_max)
        x.extend(rep_vals)
        y.extend(neighbour_vals)
        POS.extend(pos)

    # store exactly what gets plotted
    old_adj[block_names[i]] = {
        "x": np.array(x),
        "y": np.array(y),
        "POS": POS,
        "block": block,
    }

    ax.scatter(x, y, s=8, color='grey', alpha=0.5, zorder=0)

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    abline(slope, intercept, ax=ax)

    ax.text(0.4, 1.01, block_names[i], transform=ax.transAxes,
            fontsize=14, va='bottom', ha='left')

    ax.text(0.95, 0.05, 'R=%.2f' % r_value, transform=ax.transAxes,
            fontsize=7, va='bottom', ha='right')

    ax.text(0.05, 0.95, 'N=%.0f' % len(x), transform=ax.transAxes,
            fontsize=7, va='top', ha='left')

    dx, dy = np.array(x) - 1, np.array(y) - 1
    best = [dx[j]**1.3 * dy[j] if dx[j] > 0 and dy[j] > 0 else 0 for j in range(len(dx))]
    spill_i = np.argmax(best)

    ax.text(x[spill_i] + 0.01, y[spill_i], 'row/col:\n' + str(POS[spill_i]),
            fontsize=7, va='center', ha='left')

plt.show()

print(old_adj.keys())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# -----------------------------
# choose the chip EXACTLY like the script
# -----------------------------
CHIP = 'CHIP_1'   # <-- for G_unblocked / A_unblocked / ...
# CHIP = 'CHIP_2' # <-- for G_low / A_low / ...

if CHIP == 'CHIP_1':
    columns = ['unblocked', 'Song2016']
    filename = 'chip_scan.txt'
if CHIP == 'CHIP_2':
    columns = ['low', 'high']
    filename = 'chip_scan.txt'

new_path = filename

def add_dict_branch(d, keys, value):
    cur = d
    for k in keys[:-1]:
        if k not in cur:
            cur[k] = {}
        cur = cur[k]
    if keys[-1] not in cur:
        cur[keys[-1]] = value

def abline(slope, intercept, color='blue', ax='standard', alpha=0.5):
    if ax == 'standard':
        ax = plt.gca()
    x_vals = np.array(ax.get_xlim())
    y_vals = intercept + slope * x_vals
    ax.plot(x_vals, y_vals, '--', color='black', alpha=alpha, linewidth=1)

def fig_n_grid(width=9, grid_spec=[4, 2], aspect_ratio=2/3,
               left=0.09, right=0.97, bottom=0.05, top=0.97,
               wspace=0.25, hspace='standard', dpi=300):
    if hspace == 'standard':
        hspace = (wspace / aspect_ratio**(3/4))
    panel_w = width * (right-left) * (1-((grid_spec[1]-1)/grid_spec[1])*wspace) / grid_spec[1]
    height = panel_w * aspect_ratio / (1-((grid_spec[0]-1)/grid_spec[0])*hspace) * grid_spec[0] / (top-bottom)
    fig = plt.figure(figsize=[width, height], dpi=dpi)
    grid = fig.add_gridspec(grid_spec[0], grid_spec[1],
                            left=left, right=right, bottom=bottom, top=top,
                            wspace=wspace, hspace=hspace)
    return fig, grid

# -----------------------------
# original adjacency section
# -----------------------------
min_pos, max_pos = 1, 78

pos_to_F635 = {}
pos_to_ID = {}
ID_to_pos = {}
ID_to_F635 = {}

with open(new_path, 'rb') as file:
    for i, l in enumerate(file):
        try:
            l = l.decode('ascii')
        except:
            continue

        l = l.strip()
        l = l.split('\t')

        if i == 32:
            header = l
            block = header.index('"Block"')
            ID = header.index('"ID"')
            F635 = header.index('"F635 Median"')
            F532 = header.index('"F532 Median"')
            column = header.index('"Column"')
            row = header.index('"Row"')

        elif i > 32:
            add_dict_branch(pos_to_F635, [int(l[block]), int(l[row]), int(l[column])], float(l[F635]))
            add_dict_branch(pos_to_ID, [int(l[block]), int(l[row]), int(l[column])], l[ID])
            add_dict_branch(ID_to_pos, [int(l[block]), l[ID]], [])
            ID_to_pos[int(l[block])][l[ID]].append([int(l[row]), int(l[column])])
            add_dict_branch(ID_to_F635, [int(l[block]), l[ID]], [])
            ID_to_F635[int(l[block])][l[ID]].append(float(l[F635]))

# bottom 10% of IDs
lowest_per_block = {}
for block in range(1, 9):
    Is = []
    for ID in ID_to_F635[block].keys():
        I = np.mean(ID_to_F635[block][ID])
        Is.append([I, ID])
    Is.sort()
    lowest_per_block[block] = [Is[i][1] for i in range(len(Is) // 10)]

def replicates_neighbors(block, ID, mean_max='mean'):
    def surround(block, row, col):
        vals = []
        for r in [row - 1, row, row + 1]:
            for c in [col - 1, col, col + 1]:
                if not (r == row and c == col):
                    vals.append(pos_to_F635[block][r][c])
        return vals

    values, neighbors, POSs = [], [], []
    for row, col in ID_to_pos[block][ID]:
        if 1 < col < 78 and 1 < row < 78:
            values.append(pos_to_F635[block][row][col])
            neighbors.append(surround(block, row, col))
            POSs.append([row, col])

    if mean_max == 'mean':
        neighbors = [np.mean(n) for n in neighbors]
        neighbors = [n / np.mean(neighbors) for n in neighbors]
    if mean_max == 'max':
        neighbors = [max(n) for n in neighbors]
        neighbors = [n / np.mean(neighbors) for n in neighbors]
    values = [v / np.mean(values) for v in values]

    return values, neighbors, POSs

# -----------------------------
# original plotting block
# -----------------------------
mean_max = 'mean'
approach = 'all'

fig, grid = fig_n_grid(width=9, grid_spec=[4, 2], aspect_ratio=2/3, dpi=300)

ax1 = plt.subplot(grid[0, 0]); ax2 = plt.subplot(grid[0, 1])
ax3 = plt.subplot(grid[1, 0]); ax4 = plt.subplot(grid[1, 1])
ax5 = plt.subplot(grid[2, 0]); ax6 = plt.subplot(grid[2, 1])
ax7 = plt.subplot(grid[3, 0]); ax8 = plt.subplot(grid[3, 1])

column_names = [nb + C for C in ['_' + columns[0], '_' + columns[1]] for nb in 'GACU']

new_adj = {}

for i in range(8):
    block = i + 1
    ax = eval('ax' + str(i + 1))

    ax.set_xlabel('F635 / replicate-mean')
    ax.set_ylabel(mean_max + ' adjacent F635\n(rel. adjacents of replicates)')

    x, y, POS = [], [], []

    if approach == 'all':
        IDs = lowest_per_block[block]
    if approach == 'EMPTY':
        IDs = ['EMPTY']

    for ID in IDs:
        rep_vals, neighbour_vals, pos = replicates_neighbors(block, ID, mean_max=mean_max)
        x.extend(rep_vals)
        y.extend(neighbour_vals)
        POS.extend(pos)

    # store exactly what gets plotted
    new_adj[column_names[i]] = {
        "x": np.array(x),
        "y": np.array(y),
        "POS": POS,
        "block": block,
    }

    ax.scatter(x, y, s=8, color='grey', alpha=0.5, zorder=0)

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    abline(slope, intercept, ax=ax)

    ax.text(0.4, 1.01, column_names[i], transform=ax.transAxes,
            fontsize=14, va='bottom', ha='left')

    ax.text(0.95, 0.05, 'R=%.2f' % r_value, transform=ax.transAxes,
            fontsize=7, va='bottom', ha='right')

    ax.text(0.05, 0.95, 'N=%.0f' % len(x), transform=ax.transAxes,
            fontsize=7, va='top', ha='left')

    dx, dy = np.array(x) - 1, np.array(y) - 1
    best = [dx[j] ** 1.3 * dy[j] if dx[j] > 0 and dy[j] > 0 else 0 for j in range(len(dx))]
    spill_i = np.argmax(best)

    ax.text(x[spill_i] + 0.01, y[spill_i], 'row/col:\n' + str(POS[spill_i]),
            fontsize=7, va='center', ha='left')

plt.show()

print(new_adj.keys())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(9, 7), dpi=300)
axes = axes.ravel()

pairs = [
    ("G", "G_low", "G_unblocked"),
    ("A", "A_low", "A_unblocked"),
    ("C", "C_low", "C_unblocked"),
    ("U", "U_low", "U_unblocked"),
]

for i, (nt, old_key, new_key) in enumerate(pairs):
    ax = axes[i]

    x_old = old_adj[old_key]["x"]
    y_old = old_adj[old_key]["y"]

    x_new = new_adj[new_key]["x"]
    y_new = new_adj[new_key]["y"]

    # panel label
    ax.text(-0.14, 1.06, "ABCD"[i],
            transform=ax.transAxes,
            fontsize=16, fontweight="bold",
            va="top", ha="right")

    # scatter
    ax.scatter(x_old, y_old, s=8, color="grey", alpha=0.45, label=old_key, zorder=0)
    ax.scatter(x_new, y_new, s=8, color="red", alpha=0.25, label=new_key, zorder=0)

    # regression: old chip
    slope_old, intercept_old, r_old, _, _ = stats.linregress(x_old, y_old)
    xx_old = np.array(ax.get_xlim())
    ax.plot(xx_old, intercept_old + slope_old * xx_old,
        "--", color="grey", linewidth=2.5)

    # regression: new chip
    slope_new, intercept_new, r_new, _, _ = stats.linregress(x_new, y_new)
    xx_new = np.array(ax.get_xlim())
    ax.plot(xx_new, intercept_new + slope_new * xx_new,
        "--", color="red", linewidth=1.3, alpha=0.6)

    ax.set_title(nt)
    ax.set_xlabel("F635 / replicate-mean")
    ax.set_ylabel("mean adjacent F635\n(rel. adjacents of replicates)")

    ax.text(0.03, 0.97, f"{old_key}: N={len(x_old)}",
            transform=ax.transAxes,
            fontsize=8, va="top", ha="left", color="grey")

    ax.text(0.03, 0.89, f"{new_key}: N={len(x_new)}",
            transform=ax.transAxes,
            fontsize=8, va="top", ha="left", color="red")

    ax.text(0.97, 0.03,
            f"{old_key}: R={r_old:.2f}\n{new_key}: R={r_new:.2f}",
            transform=ax.transAxes,
            fontsize=8, va="bottom", ha="right")

    if i == 0:
        ax.legend(frameon=False, fontsize=8, loc="upper right")

plt.tight_layout()
plt.savefig("adjacency_old_vs_new_combined.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import defaultdict

# ========= INPUT =========
old_path = "chip_scan.txt"

# ========= LOAD RAW FILE =========
pos_to_F635 = defaultdict(lambda: defaultdict(dict))
pos_to_ID   = defaultdict(lambda: defaultdict(dict))
ID_to_pos   = defaultdict(lambda: defaultdict(list))
ID_to_F635  = defaultdict(lambda: defaultdict(list))

with open(old_path, "rb") as file:
    for i, l in enumerate(file):
        try:
            l = l.decode("ascii")
        except:
            continue

        l = l.strip().split("\t")

        if i == 27:
            header = l
            block_i  = header.index('"Block"')
            ID_i     = header.index('"ID"')
            F635_i   = header.index('"F635 Median"')
            F532_i   = header.index('"F532 Median"')
            column_i = header.index('"Column"')
            row_i    = header.index('"Row"')

        elif i > 27:
            block = int(l[block_i])
            row   = int(l[row_i])
            col   = int(l[column_i])
            ID    = l[ID_i]
            F635  = float(l[F635_i])

            pos_to_F635[block][row][col] = F635
            pos_to_ID[block][row][col] = ID
            ID_to_pos[block][ID].append((row, col))
            ID_to_F635[block][ID].append(F635)

# ========= LOWEST 10% IDS =========
lowest_per_block = {}
for block in range(1, 9):
    Is = []
    for ID in ID_to_F635[block].keys():
        I = np.mean(ID_to_F635[block][ID])
        Is.append((I, ID))
    Is.sort()
    lowest_per_block[block] = [Is[i][1] for i in range(len(Is)//10)]

# ========= SAME NEIGHBOR LOGIC =========
def replicates_neighbors(block, ID, mean_max="mean"):
    def surround(block, row, col):
        vals = []
        for r in [row-1, row, row+1]:
            for c in [col-1, col, col+1]:
                if not (r == row and c == col):
                    vals.append(pos_to_F635[block][r][c])
        return vals

    values, neighbors, POSs = [], [], []

    for row, col in ID_to_pos[block][ID]:
        if 1 < col < 78 and 1 < row < 78:
            values.append(pos_to_F635[block][row][col])
            neighbors.append(surround(block, row, col))
            POSs.append((row, col))

    if mean_max == "mean":
        neighbors = [np.mean(n) for n in neighbors]
        neighbors = [n / np.mean(neighbors) for n in neighbors]
    elif mean_max == "max":
        neighbors = [max(n) for n in neighbors]
        neighbors = [n / np.mean(neighbors) for n in neighbors]

    values = [v / np.mean(values) for v in values]

    return values, neighbors, POSs

# ========= HELPER =========
def abline(slope, intercept, ax, alpha=0.5):
    x_vals = np.array(ax.get_xlim())
    y_vals = intercept + slope * x_vals
    ax.plot(x_vals, y_vals, "--", color="black", alpha=alpha, linewidth=1)

# ========= SETTINGS =========
mean_max = "mean"
approach = "EMPTY"   # change to "all" for lowest 10% IDs

block_names = [nb + C for C in ["_low", "_high"] for nb in "GACU"]

# ========= PLOT =========
fig, axes = plt.subplots(4, 2, figsize=(9, 14), dpi=300)
axes = axes.ravel()

old_adj = {}

for i in range(8):
    block = i + 1
    ax = axes[i]

    ax.set_xlabel("F635 / replicate-mean")
    ax.set_ylabel(mean_max + " adjacent F635\n(rel. adjacents of replicates)")

    x, y, POS = [], [], []

    if approach == "all":
        IDs = lowest_per_block[block]
    elif approach == "EMPTY":
        IDs = ["EMPTY"]
    else:
        raise ValueError("approach must be 'all' or 'EMPTY'")

    for ID in IDs:
        if ID not in ID_to_pos[block]:
            print(f"Missing ID {ID} in block {block} ({block_names[i]})")
            continue

        rep_vals, neighbour_vals, pos = replicates_neighbors(block, ID, mean_max=mean_max)
        x.extend(rep_vals)
        y.extend(neighbour_vals)
        POS.extend(pos)

    x = np.array(x)
    y = np.array(y)

    old_adj[block_names[i]] = {
        "x": x,
        "y": y,
        "POS": POS,
        "block": block,
        "IDs": IDs,
    }

    ax.scatter(x, y, s=8, color="grey", alpha=0.5, zorder=0)

    if len(x) >= 2:
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        abline(slope, intercept, ax=ax)

        ax.text(0.95, 0.05, f"R={r_value:.2f}",
                transform=ax.transAxes, fontsize=7, va="bottom", ha="right")
    else:
        ax.text(0.95, 0.05, "R=NA",
                transform=ax.transAxes, fontsize=7, va="bottom", ha="right")

    ax.text(0.4, 1.01, block_names[i],
            transform=ax.transAxes, fontsize=14, va="bottom", ha="left")

    ax.text(0.05, 0.95, f"N={len(x):.0f}",
            transform=ax.transAxes, fontsize=7, va="top", ha="left")

    if len(x) > 0:
        dx, dy = x - 1, y - 1
        best = [dx[j]**1.3 * dy[j] if dx[j] > 0 and dy[j] > 0 else 0 for j in range(len(dx))]
        spill_i = int(np.argmax(best))
        ax.text(x[spill_i] + 0.01, y[spill_i], f"row/col:\n{POS[spill_i]}",
                fontsize=7, va="center", ha="left")

plt.tight_layout()
plt.show()

print(old_adj.keys())
for k in block_names:
    print(k, "N =", len(old_adj[k]["x"]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(9, 7), dpi=300)
axes = axes.ravel()

pairs = [
    ("G", "G_low", "G_high"),
    ("A", "A_low", "A_high"),
    ("C", "C_low", "C_high"),
    ("U", "U_low", "U_high"),
]

for i, (nt, low_key, high_key) in enumerate(pairs):
    ax = axes[i]

    x_low = old_adj[low_key]["x"]
    y_low = old_adj[low_key]["y"]

    x_high = old_adj[high_key]["x"]
    y_high = old_adj[high_key]["y"]

    ax.text(-0.14, 1.06, "ABCD"[i],
            transform=ax.transAxes,
            fontsize=16, fontweight="bold",
            va="top", ha="right")

    # scatter
    ax.scatter(x_low, y_low, s=8, color="grey", alpha=0.45, label="low", zorder=0)
    ax.scatter(x_high, y_high, s=8, color="red", alpha=0.28, label="high", zorder=0)

    # regression low
    slope_low, intercept_low, r_low, _, _ = stats.linregress(x_low, y_low)
    xx_low = np.array(ax.get_xlim())
    ax.plot(xx_low, intercept_low + slope_low * xx_low,
            "--", color="black", linewidth=2)

    # regression high
    slope_high, intercept_high, r_high, _, _ = stats.linregress(x_high, y_high)
    xx_high = np.array(ax.get_xlim())
    ax.plot(xx_high, intercept_high + slope_high * xx_high,
            "--", color="red", linewidth=1.2, alpha=0.7)

    ax.set_title(nt)
    ax.set_xlabel("F635 / replicate-mean")
    ax.set_ylabel("mean adjacent F635\n(rel. adjacents of replicates)")

    ax.text(0.03, 0.97, f"low N={len(x_low)}",
            transform=ax.transAxes,
            fontsize=8, va="top", ha="left", color="grey")

    ax.text(0.03, 0.89, f"high N={len(x_high)}",
            transform=ax.transAxes,
            fontsize=8, va="top", ha="left", color="red")

    ax.text(0.97, 0.03,
            f"low R={r_low:.2f}\nhigh R={r_high:.2f}",
            transform=ax.transAxes,
            fontsize=8, va="bottom", ha="right")

    if i == 0:
        ax.legend(frameon=False, fontsize=8, loc="upper right")

plt.tight_layout()
plt.savefig("adjacency_oldchip_low_high_combined.png", dpi=300)
plt.show()

<a id="sec3"></a>
## 3. Rebuild human-readable scales table

Regenerates a human-readable amino-acid scales table (per scale and nucleotide) from a stored chip-scales pickle using the same additive model as the main pipeline.

<sub>Source script: <code>rebuild_scales_table.ipynb</code></sub>

In [ ]:
import pickle
chip_scales_mono = {scale: {} for scale in ['mo', 'dimo', 'trimo', 'decamo']}
chip_scales = {}

In [ ]:
with open("chip_scales.data", 'rb') as f:
    chip_scales = pickle.load(f)
display(chip_scales)

In [ ]:
# make_scales_from_chip_scales.py
# Rebuilds scales.txt (mo/dimo/trimo/decamo per AA, per nucleotide)
# from a 2024 chip_scales.data pickle using the same additive-model
# approach as in chip_march_2024.py.
#
# Input:  chip_scales.data  (dict: { 'G'|'A'|'C'|'U' : {peptide:str -> mean_intensity:float} })
# Output: scales.txt        (human-readable AA tables by scale and nucleotide)

import os
import pickle
import re
import numpy as np
from scipy.optimize import lsq_linear

# ==== CONFIG ====
AA_ORDER = 'ARNDCQEGHILKMFPSTWYV'
AA_SET = set(AA_ORDER)
NUCS = 'GACU'

# Set your paths:
# - in_path: the 2024 chip_scales.data you uploaded / generated
# - out_txt: where scales.txt should be written
in_path = "chip_scales.data"
out_txt = "scales.txt"

# Optionally also save a pickle with the computed mono scales:
# optional_chip_scales_mono_pickle = "/path/to/your/2024/CHIP_2/scales/chip_scales_mono_from_chip_scales.data"

# ==== HELPERS ====

def encode_20(seq: str) -> np.ndarray:
    """Count vector for AA_ORDER."""
    return np.array([seq.count(aa) for aa in AA_ORDER], dtype=float)

def is_valid_pep(seq: str) -> bool:
    """Peptide letters are standard 20 AAs only."""
    return len(seq) > 0 and set(seq).issubset(AA_SET)

def is_homo10(seq: str) -> bool:
    return len(seq) == 10 and len(set(seq)) == 1 and is_valid_pep(seq)

def is_dimer_repeat5(seq: str) -> bool:
    """(AB)*5, not homo."""
    if len(seq) != 10 or not is_valid_pep(seq):
        return False
    a_odd = set(seq[0::2])     # positions 0,2,4,6,8
    a_even = set(seq[1::2])    # positions 1,3,5,7,9
    if len(a_odd) == 1 and len(a_even) == 1:
        a = next(iter(a_odd))
        b = next(iter(a_even))
        return a != b
    return False

def is_tri11_pattern(seq: str) -> bool:
    """tri + 'G' + tri + 'G' + tri (length 11), as used in the scripts."""
    if len(seq) != 11:
        return False
    if not is_valid_pep(seq):
        return False
    tri = seq[:3]
    return seq == (tri + 'G' + tri + 'G' + tri)

def is_decamer(seq: str) -> bool:
    """Natural 10-mers used for the 10-mer fit, excluding homo and dimer."""
    return len(seq) == 10 and is_valid_pep(seq) and (not is_homo10(seq)) and (not is_dimer_repeat5(seq))

def fit_additive(peptides, values, bounds=(-10000, 100000)):
    """Fit AA contributions with BVLS: y ≈ X * beta, X = 20-AA counts."""
    if len(peptides) == 0:
        return np.array([np.nan]*20)
    X = np.vstack([encode_20(p) for p in peptides])
    y = np.array(values, dtype=float)
    # Robustness: if everything is identical, lsq might be unhappy
    res = lsq_linear(X, y, bounds=bounds, method="bvls")
    return res.x

# ==== LOAD ====
with open(in_path, "rb") as f:
    chip_scales = pickle.load(f)
# Expect: chip_scales['G'|'A'|'C'|'U'][peptide] = mean_intensity

# ==== BUILD mo/dimo/trimo/decamo ====
chip_scales_mono = {k: {nb: {} for nb in NUCS} for k in ["mo", "dimo", "trimo", "decamo"]}

for nb in NUCS:
    if nb not in chip_scales:
        continue

    # Collect peptide sets by class
    homo_peps, homo_vals = [], []
    dimer_peps, dimer_vals = [], []
    tri_peps, tri_vals = [], []
    deca_peps, deca_vals = [], []

    for pep, val in chip_scales[nb].items():
        if not is_valid_pep(pep):
            continue
        if is_homo10(pep):
            homo_peps.append(pep)
            homo_vals.append(val)
        elif is_dimer_repeat5(pep):
            dimer_peps.append(pep)
            dimer_vals.append(val)
        elif is_tri11_pattern(pep):
            tri_peps.append(pep)
            tri_vals.append(val)
        elif is_decamer(pep):
            deca_peps.append(pep)
            deca_vals.append(val)
        # else: ignore other constructs/fillers, e.g. '####'

    # --- mo: per-AA value from homopolymers (mean(AA*10)/10) ---
    # If multiple replicates exist, average; if missing AA, leave as NaN/skip.
    for aa in AA_ORDER:
        homo10 = aa * 10
        vals = [v for p, v in zip(homo_peps, homo_vals) if p == homo10]
        chip_scales_mono["mo"][nb][aa] = round(float(np.mean(vals)) / 10, 2) if len(vals) else np.nan

    # --- dimo: fit on (AB)5 set ---
    beta_dimo = fit_additive(dimer_peps, dimer_vals, bounds=(-10000, 100000))
    for i, aa in enumerate(AA_ORDER):
        chip_scales_mono["dimo"][nb][aa] = round(float(beta_dimo[i]), 2)

    # --- trimo: fit on tri+'G'+tri+'G'+tri (11-mers) ---
    beta_trimo = fit_additive(tri_peps, tri_vals, bounds=(-10000, 100000))
    for i, aa in enumerate(AA_ORDER):
        chip_scales_mono["trimo"][nb][aa] = round(float(beta_trimo[i]), 2)

    # --- decamo: fit on natural 10-mers (not homo/dimer) ---
    beta_decamo = fit_additive(deca_peps, deca_vals, bounds=(-10000, 100000))
    for i, aa in enumerate(AA_ORDER):
        chip_scales_mono["decamo"][nb][aa] = round(float(beta_decamo[i]), 2)

# ==== WRITE scales.txt (same style as before) ====
lines = []
lines.append("# Derived amino-acid scales from 2024 chip_scales.data\n")
lines.append("# Scales: mo (homopolymer/10), dimo ((AB)5 fit), trimo (tri+G+tri+G+tri fit), decamo (10-mers fit)\n\n")

for scale_name in ["mo", "dimo", "trimo", "decamo"]:
    lines.append(f"=== {scale_name} ===")
    for nb in NUCS:
        if nb not in chip_scales_mono[scale_name]:
            continue
        lines.append(f"[{nb}]")
        for aa in AA_ORDER:
            val = chip_scales_mono[scale_name][nb].get(aa, np.nan)
            sval = "NA" if (val is None or (isinstance(val, float) and np.isnan(val))) else f"{val:.2f}"
            lines.append(f"{aa}\t{sval}")
        lines.append("")  # blank line
    lines.append("")      # extra space between scales

with open(out_txt, "w") as f:
    f.write("\n".join(lines))

# (Optional) also store the computed dict for reuse
try:
    with open(optional_chip_scales_mono_pickle, "wb") as f:
        pickle.dump(chip_scales_mono, f)
except Exception:
    pass

print("Wrote:", out_txt)


In [ ]:
import os
import pickle
import numpy as np
from scipy.optimize import lsq_linear

# ========== Configuration ==========
AA_ORDER = 'ARNDCQEGHILKMFPSTWYV'
AAs = list(AA_ORDER)
homos = set([aa * 10 for aa in AAs])
BIs = set([(a + b) * 5 for a in AAs for b in AAs])

# Path to your 2024 chip_scales.data
in_path = "chip_scales.data"
# Where to save the new additive scales
out_path = "scales_2024_chip2.data"  # (pickle dump of dict)

# ========== Helpers ==========

def encode_20(seq):
    """Count vector for each of the 20 standard amino acids."""
    return np.array([seq.count(aa) for aa in AA_ORDER], dtype=float)

# ========== Load data ==========
with open(in_path, "rb") as f:
    chip_scales = pickle.load(f)

# ========== Fit linear additive model ==========
# Mirrors your provided snippet exactly
chip_scales_fitted = {}

column_names = [nb + C for C in ['_low', '_high'] for nb in 'GACU']

for block in chip_scales.keys():
    X, y = [], []

    for peptide in BIs - homos:
        if peptide not in chip_scales[block]:
            continue
        x = encode_20(peptide)
        X.append(x)
        y.append(np.mean(chip_scales[block][peptide]))

    if len(X) == 0:
        print(f"⚠️  Block {block} has no valid (AB)₅ peptides.")
        continue

    X = np.array(X)
    y = np.array(y)

    # BVLS additive fit (same bounds as your snippet)
    res = lsq_linear(X, y, bounds=(-10000, 10000), method='bvls')

    # Save fitted amino acid coefficients
    chip_scales_fitted[block] = {AAs[i]: round(float(res.x[i]), 3) for i in range(20)}

    print(f"✅ Fitted block {block} — R^2 ≈ {1 - np.var(y - X.dot(res.x)) / np.var(y):.3f}")

# ========== Save to .txt (as pickle dump) ==========
# Note: using pickle.dump as you requested
with open(out_path, "wb") as f:
    pickle.dump(chip_scales_fitted, f)

print(f"\n✅ Saved fitted scales (additive model) to:\n{out_path}")


In [ ]:
with open("scales_2024_chip2.data", 'rb') as e:
    chip_scales_2024 = pickle.load(e)
display(chip_scales_2024)

<a id="sec4"></a>
## 4. Binding-scale visualisation

Visualises the derived amino-acid binding scales (mono/di/tri/deca) across nucleotides as grouped plots and heatmaps.

<sub>Source script: <code>binding_scales_plots.ipynb</code></sub>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data = {'mo': {'G': {'A': 183.39,
   'R': 263.51,
   'N': 322.84,
   'D': 204.21,
   'C': 207.82,
   'Q': 207.25,
   'E': 226.48,
   'G': 247.36,
   'H': 267.35,
   'I': 298.52,
   'L': 263.75,
   'K': 1622.8,
   'M': 176.15,
   'F': 383.75,
   'P': 605.89,
   'S': 268.68,
   'T': 205.88,
   'W': 1216.15,
   'Y': 1395.1,
   'V': 358.3},
  'A': {'A': 359.81,
   'R': 535.29,
   'N': 397.14,
   'D': 407.16,
   'C': 571.46,
   'Q': 340.1,
   'E': 473.09,
   'G': 393.22,
   'H': 481.7,
   'I': 557.26,
   'L': 631.24,
   'K': 336.84,
   'M': 403.98,
   'F': 897.19,
   'P': 624.48,
   'S': 358.51,
   'T': 405.48,
   'W': 1363.62,
   'Y': 1317.88,
   'V': 601.99},
  'C': {'A': 120.99,
   'R': 108.92,
   'N': 166.32,
   'D': 315.23,
   'C': 255.26,
   'Q': 145.98,
   'E': 342.34,
   'G': 155.11,
   'H': 132.88,
   'I': 116.39,
   'L': 153.16,
   'K': 106.46,
   'M': 149.98,
   'F': 168.9,
   'P': 165.44,
   'S': 136.48,
   'T': 157.81,
   'W': 354.74,
   'Y': 258.6,
   'V': 147.78},
  'U': {'A': 72.1,
   'R': 67.54,
   'N': 82.16,
   'D': 131.99,
   'C': 149.34,
   'Q': 76.44,
   'E': 146.85,
   'G': 82.49,
   'H': 77.74,
   'I': 68.46,
   'L': 77.69,
   'K': 69.32,
   'M': 80.22,
   'F': 91.54,
   'P': 87.52,
   'S': 83.4,
   'T': 86.04,
   'W': 208.62,
   'Y': 139.89,
   'V': 81.1}},
 'dimo': {'G': {'A': 303.52,
   'R': 154.09,
   'N': 341.84,
   'D': 113.2,
   'C': 51.05,
   'Q': 298.13,
   'E': 162.52,
   'G': 302.67,
   'H': 249.18,
   'I': 256.3,
   'L': 321.64,
   'K': 671.61,
   'M': 151.8,
   'F': 553.62,
   'P': 451.8,
   'S': 373.2,
   'T': 281.23,
   'W': 959.18,
   'Y': 924.99,
   'V': 311.98},
  'A': {'A': 431.08,
   'R': 366.82,
   'N': 386.96,
   'D': 461.52,
   'C': 553.17,
   'Q': 337.06,
   'E': 567.43,
   'G': 430.83,
   'H': 476.14,
   'I': 559.23,
   'L': 686.09,
   'K': 138.35,
   'M': 379.27,
   'F': 915.43,
   'P': 579.9,
   'S': 417.94,
   'T': 433.74,
   'W': 1271.05,
   'Y': 1031.41,
   'V': 578.5},
  'C': {'A': 143.0,
   'R': 46.61,
   'N': 141.27,
   'D': 523.78,
   'C': 291.95,
   'Q': 136.14,
   'E': 604.21,
   'G': 163.48,
   'H': 135.45,
   'I': 123.5,
   'L': 183.18,
   'K': -29.92,
   'M': 141.88,
   'F': 225.96,
   'P': 153.74,
   'S': 144.78,
   'T': 158.38,
   'W': 321.55,
   'Y': 242.68,
   'V': 153.46},
  'U': {'A': 79.98,
   'R': 44.54,
   'N': 77.94,
   'D': 205.76,
   'C': 169.9,
   'Q': 74.7,
   'E': 240.28,
   'G': 89.2,
   'H': 70.56,
   'I': 68.58,
   'L': 97.58,
   'K': 21.14,
   'M': 77.77,
   'F': 121.17,
   'P': 80.58,
   'S': 79.49,
   'T': 89.0,
   'W': 181.44,
   'Y': 142.89,
   'V': 84.83}},
 'trimo': {'G': {'A': 262.7,
   'R': 163.08,
   'N': 251.77,
   'D': 44.94,
   'C': 74.19,
   'Q': 170.88,
   'E': 121.31,
   'G': 235.21,
   'H': 252.86,
   'I': 328.14,
   'L': 377.43,
   'K': 486.23,
   'M': 115.13,
   'F': 582.42,
   'P': 327.95,
   'S': 282.49,
   'T': 245.88,
   'W': 683.35,
   'Y': 766.39,
   'V': 320.03},
  'A': {'A': 378.36,
   'R': 295.0,
   'N': 332.35,
   'D': 540.27,
   'C': 534.99,
   'Q': 298.09,
   'E': 639.16,
   'G': 356.92,
   'H': 487.19,
   'I': 530.61,
   'L': 646.81,
   'K': 16.7,
   'M': 363.1,
   'F': 817.83,
   'P': 451.34,
   'S': 362.23,
   'T': 406.25,
   'W': 1059.72,
   'Y': 845.74,
   'V': 552.55},
  'C': {'A': 132.3,
   'R': 0.98,
   'N': 119.09,
   'D': 593.62,
   'C': 317.15,
   'Q': 119.16,
   'E': 644.5,
   'G': 137.82,
   'H': 113.14,
   'I': 138.2,
   'L': 219.46,
   'K': -122.0,
   'M': 137.28,
   'F': 267.93,
   'P': 129.29,
   'S': 140.37,
   'T': 163.81,
   'W': 344.74,
   'Y': 245.81,
   'V': 168.01},
  'U': {'A': 73.98,
   'R': 28.27,
   'N': 66.34,
   'D': 228.92,
   'C': 163.38,
   'Q': 68.12,
   'E': 253.59,
   'G': 76.89,
   'H': 52.96,
   'I': 73.55,
   'L': 112.47,
   'K': -15.8,
   'M': 71.59,
   'F': 133.78,
   'P': 69.37,
   'S': 77.58,
   'T': 87.21,
   'W': 181.61,
   'Y': 139.11,
   'V': 89.03}},
 'decamo': {'G': {'A': 313.65,
   'R': 171.15,
   'N': 317.19,
   'D': -123.94,
   'C': -113.76,
   'Q': 146.56,
   'E': -26.97,
   'G': 280.75,
   'H': 666.91,
   'I': 480.04,
   'L': 299.41,
   'K': 252.01,
   'M': 345.67,
   'F': 446.73,
   'P': 437.38,
   'S': 267.45,
   'T': 177.94,
   'W': 650.79,
   'Y': 793.0,
   'V': 330.71},
  'A': {'A': 529.53,
   'R': 345.91,
   'N': 346.24,
   'D': 511.93,
   'C': 770.08,
   'Q': 280.72,
   'E': 618.56,
   'G': 379.76,
   'H': 846.57,
   'I': 624.42,
   'L': 563.42,
   'K': -164.46,
   'M': 564.84,
   'F': 757.32,
   'P': 471.08,
   'S': 383.09,
   'T': 434.98,
   'W': 914.72,
   'Y': 900.34,
   'V': 506.41},
  'C': {'A': 171.14,
   'R': 0.75,
   'N': 147.18,
   'D': 528.71,
   'C': 275.36,
   'Q': 124.41,
   'E': 757.33,
   'G': 151.8,
   'H': 185.13,
   'I': 227.99,
   'L': 82.59,
   'K': -295.56,
   'M': 281.64,
   'F': 248.61,
   'P': 190.03,
   'S': 170.08,
   'T': 216.06,
   'W': 303.75,
   'Y': 185.13,
   'V': 180.41},
  'U': {'A': 94.0,
   'R': 26.93,
   'N': 74.65,
   'D': 204.88,
   'C': 154.05,
   'Q': 75.26,
   'E': 301.46,
   'G': 85.53,
   'H': 95.38,
   'I': 113.69,
   'L': 71.99,
   'K': -79.01,
   'M': 130.84,
   'F': 138.15,
   'P': 98.26,
   'S': 87.79,
   'T': 103.92,
   'W': 183.3,
   'Y': 119.81,
   'V': 91.13}}}


In [ ]:
# Labels (AA codes)
labels = list(next(iter(data.values()))['G'].keys())  # e.g., ['A', 'R', 'N', ..., 'V']
rows = list(data.keys())  # ['mo', 'dimo', 'trimo', 'decamo']
sources = ['G', 'A', 'C', 'U']

# Build 4x80 matrix (4 rows, 20 AAs * 4 conditions)
matrix = []
for row in rows:
    row_values = []
    for src in sources:
        for aa in labels:
            value = data[row][src].get(aa, np.nan)
            row_values.append(value)
    matrix.append(row_values)

matrix = np.array(matrix)

# Normalize for colormap
min_val = np.nanmin(matrix)
max_val = np.nanmax(matrix)
norm = (matrix - min_val) / (max_val - min_val)

# Create plot
fig, ax = plt.subplots(figsize=(22, 4))
ax.set_axis_off()

# Generate column labels like A-G, R-G, ..., V-U
col_labels = [f"{aa}-{src}" for src in sources for aa in labels]

# Create the table
table = ax.table(cellText=None,
                 cellColours=plt.cm.Reds(norm),
                 colLabels=col_labels,
                 rowLabels=rows,
                 loc='center',
                 cellLoc='center')

# Add values as text in the cells
for i in range(len(rows)):
    for j in range(len(col_labels)):
        val = matrix[i, j]
        table[i+1, j].get_text().set_text(f"{val:.1f}")

plt.tight_layout()
plt.show()


In [ ]:
aa_labels = list(data['mo']['G'].keys())  # 20 amino acids
sources = ['G', 'A', 'C', 'U']
rows = list(data.keys())  # ['mo', 'dimo', 'trimo', 'decamo']

# Set up global color scaling across all values
all_values = []
for row in rows:
    for src in sources:
        all_values.extend(data[row][src].values())

min_val = min(all_values)
max_val = max(all_values)

# Generate one 20x4 heatmap per row (mo, dimo, ...)
for row in rows:
    heatmap_data = []
    for aa in aa_labels:
        row_vals = [data[row][src][aa] for src in sources]
        heatmap_data.append(row_vals)

    heatmap_data = np.array(heatmap_data)

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(6, 10))
    # Use vmin and vmax to set the full value range for the colormap
    im = ax.imshow(heatmap_data, cmap="Reds", vmin=min_val, vmax=max_val)

    # Configure ticks
    ax.set_xticks(np.arange(len(sources)))
    ax.set_xticklabels(sources)
    ax.set_yticks(np.arange(len(aa_labels)))
    ax.set_yticklabels(aa_labels)

    # No text in cells
    ax.set_title(f"{row} Heatmap", fontsize=14)
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center", fontsize=10)
    plt.setp(ax.get_yticklabels(), fontsize=10)

    # Remove grid and ticks
    ax.tick_params(top=False, bottom=True, left=True, right=False)

    # Add colorbar and show raw values
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Raw Signal", fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np

data = {'mo': {'G': {'A': 183.39,
   'R': 263.51,
   'N': 322.84,
   'D': 204.21,
   'C': 207.82,
   'Q': 207.25,
   'E': 226.48,
   'G': 247.36,
   'H': 267.35,
   'I': 298.52,
   'L': 263.75,
   'K': 1622.8,
   'M': 176.15,
   'F': 383.75,
   'P': 605.89,
   'S': 268.68,
   'T': 205.88,
   'W': 1216.15,
   'Y': 1395.1,
   'V': 358.3},
  'A': {'A': 359.81,
   'R': 535.29,
   'N': 397.14,
   'D': 407.16,
   'C': 571.46,
   'Q': 340.1,
   'E': 473.09,
   'G': 393.22,
   'H': 481.7,
   'I': 557.26,
   'L': 631.24,
   'K': 336.84,
   'M': 403.98,
   'F': 897.19,
   'P': 624.48,
   'S': 358.51,
   'T': 405.48,
   'W': 1363.62,
   'Y': 1317.88,
   'V': 601.99},
  'C': {'A': 120.99,
   'R': 108.92,
   'N': 166.32,
   'D': 315.23,
   'C': 255.26,
   'Q': 145.98,
   'E': 342.34,
   'G': 155.11,
   'H': 132.88,
   'I': 116.39,
   'L': 153.16,
   'K': 106.46,
   'M': 149.98,
   'F': 168.9,
   'P': 165.44,
   'S': 136.48,
   'T': 157.81,
   'W': 354.74,
   'Y': 258.6,
   'V': 147.78},
  'U': {'A': 72.1,
   'R': 67.54,
   'N': 82.16,
   'D': 131.99,
   'C': 149.34,
   'Q': 76.44,
   'E': 146.85,
   'G': 82.49,
   'H': 77.74,
   'I': 68.46,
   'L': 77.69,
   'K': 69.32,
   'M': 80.22,
   'F': 91.54,
   'P': 87.52,
   'S': 83.4,
   'T': 86.04,
   'W': 208.62,
   'Y': 139.89,
   'V': 81.1}},
 'dimo': {'G': {'A': 303.52,
   'R': 154.09,
   'N': 341.84,
   'D': 113.2,
   'C': 51.05,
   'Q': 298.13,
   'E': 162.52,
   'G': 302.67,
   'H': 249.18,
   'I': 256.3,
   'L': 321.64,
   'K': 671.61,
   'M': 151.8,
   'F': 553.62,
   'P': 451.8,
   'S': 373.2,
   'T': 281.23,
   'W': 959.18,
   'Y': 924.99,
   'V': 311.98},
  'A': {'A': 431.08,
   'R': 366.82,
   'N': 386.96,
   'D': 461.52,
   'C': 553.17,
   'Q': 337.06,
   'E': 567.43,
   'G': 430.83,
   'H': 476.14,
   'I': 559.23,
   'L': 686.09,
   'K': 138.35,
   'M': 379.27,
   'F': 915.43,
   'P': 579.9,
   'S': 417.94,
   'T': 433.74,
   'W': 1271.05,
   'Y': 1031.41,
   'V': 578.5},
  'C': {'A': 143.0,
   'R': 46.61,
   'N': 141.27,
   'D': 523.78,
   'C': 291.95,
   'Q': 136.14,
   'E': 604.21,
   'G': 163.48,
   'H': 135.45,
   'I': 123.5,
   'L': 183.18,
   'K': -29.92,
   'M': 141.88,
   'F': 225.96,
   'P': 153.74,
   'S': 144.78,
   'T': 158.38,
   'W': 321.55,
   'Y': 242.68,
   'V': 153.46},
  'U': {'A': 79.98,
   'R': 44.54,
   'N': 77.94,
   'D': 205.76,
   'C': 169.9,
   'Q': 74.7,
   'E': 240.28,
   'G': 89.2,
   'H': 70.56,
   'I': 68.58,
   'L': 97.58,
   'K': 21.14,
   'M': 77.77,
   'F': 121.17,
   'P': 80.58,
   'S': 79.49,
   'T': 89.0,
   'W': 181.44,
   'Y': 142.89,
   'V': 84.83}},
 'trimo': {'G': {'A': 262.7,
   'R': 163.08,
   'N': 251.77,
   'D': 44.94,
   'C': 74.19,
   'Q': 170.88,
   'E': 121.31,
   'G': 235.21,
   'H': 252.86,
   'I': 328.14,
   'L': 377.43,
   'K': 486.23,
   'M': 115.13,
   'F': 582.42,
   'P': 327.95,
   'S': 282.49,
   'T': 245.88,
   'W': 683.35,
   'Y': 766.39,
   'V': 320.03},
  'A': {'A': 378.36,
   'R': 295.0,
   'N': 332.35,
   'D': 540.27,
   'C': 534.99,
   'Q': 298.09,
   'E': 639.16,
   'G': 356.92,
   'H': 487.19,
   'I': 530.61,
   'L': 646.81,
   'K': 16.7,
   'M': 363.1,
   'F': 817.83,
   'P': 451.34,
   'S': 362.23,
   'T': 406.25,
   'W': 1059.72,
   'Y': 845.74,
   'V': 552.55},
  'C': {'A': 132.3,
   'R': 0.98,
   'N': 119.09,
   'D': 593.62,
   'C': 317.15,
   'Q': 119.16,
   'E': 644.5,
   'G': 137.82,
   'H': 113.14,
   'I': 138.2,
   'L': 219.46,
   'K': -122.0,
   'M': 137.28,
   'F': 267.93,
   'P': 129.29,
   'S': 140.37,
   'T': 163.81,
   'W': 344.74,
   'Y': 245.81,
   'V': 168.01},
  'U': {'A': 73.98,
   'R': 28.27,
   'N': 66.34,
   'D': 228.92,
   'C': 163.38,
   'Q': 68.12,
   'E': 253.59,
   'G': 76.89,
   'H': 52.96,
   'I': 73.55,
   'L': 112.47,
   'K': -15.8,
   'M': 71.59,
   'F': 133.78,
   'P': 69.37,
   'S': 77.58,
   'T': 87.21,
   'W': 181.61,
   'Y': 139.11,
   'V': 89.03}},
 'decamo': {'G': {'A': 313.65,
   'R': 171.15,
   'N': 317.19,
   'D': -123.94,
   'C': -113.76,
   'Q': 146.56,
   'E': -26.97,
   'G': 280.75,
   'H': 666.91,
   'I': 480.04,
   'L': 299.41,
   'K': 252.01,
   'M': 345.67,
   'F': 446.73,
   'P': 437.38,
   'S': 267.45,
   'T': 177.94,
   'W': 650.79,
   'Y': 793.0,
   'V': 330.71},
  'A': {'A': 529.53,
   'R': 345.91,
   'N': 346.24,
   'D': 511.93,
   'C': 770.08,
   'Q': 280.72,
   'E': 618.56,
   'G': 379.76,
   'H': 846.57,
   'I': 624.42,
   'L': 563.42,
   'K': -164.46,
   'M': 564.84,
   'F': 757.32,
   'P': 471.08,
   'S': 383.09,
   'T': 434.98,
   'W': 914.72,
   'Y': 900.34,
   'V': 506.41},
  'C': {'A': 171.14,
   'R': 0.75,
   'N': 147.18,
   'D': 528.71,
   'C': 275.36,
   'Q': 124.41,
   'E': 757.33,
   'G': 151.8,
   'H': 185.13,
   'I': 227.99,
   'L': 82.59,
   'K': -295.56,
   'M': 281.64,
   'F': 248.61,
   'P': 190.03,
   'S': 170.08,
   'T': 216.06,
   'W': 303.75,
   'Y': 185.13,
   'V': 180.41},
  'U': {'A': 94.0,
   'R': 26.93,
   'N': 74.65,
   'D': 204.88,
   'C': 154.05,
   'Q': 75.26,
   'E': 301.46,
   'G': 85.53,
   'H': 95.38,
   'I': 113.69,
   'L': 71.99,
   'K': -79.01,
   'M': 130.84,
   'F': 138.15,
   'P': 98.26,
   'S': 87.79,
   'T': 103.92,
   'W': 183.3,
   'Y': 119.81,
   'V': 91.13}}}

amino_acids_to_extract = ['D', 'M', 'A']
rna_bases = [ 'A', 'C','G', 'U']
scales = list(data.keys())

# Prepare a list to store data for the DataFrame
data_for_df = []

print("Extracted values for Aspartate (D), Methionine (M), and Alanine (A):\n")

for scale in scales:
    print(f"Scale: {scale}")
    for aa in amino_acids_to_extract:
        print(f"  Amino Acid: {aa}")
        for base in rna_bases:
            value = data[scale][base].get(aa, np.nan)
            print(f"    RNA Base {base}: {value}")
            data_for_df.append({
                'Scale': scale,
                'Amino Acid': aa,
                'RNA Base': base,
                'Value': value
            })
    print("\n")

# Create a pandas DataFrame from the collected data
extracted_df = pd.DataFrame(data_for_df)

print("DataFrame of extracted values:\n")
print(extracted_df)

# Convert the DataFrame to a NumPy array
pivoted_df = extracted_df.pivot_table(index=['Scale', 'Amino Acid'], columns='RNA Base', values='Value')
numpy_array = pivoted_df.values

print("\nNumPy array of extracted values:\n")
print(numpy_array)
print("\nShape of the NumPy array:")
print(numpy_array.shape)

In [ ]:
data = {
    "Asp": {
        "Nucleotide": ["AMP","CMP","GMP","UMP"],
        "Dia":        [-5.0522, -6.9774, -4.3174, -6.5701],
        "mo_oligo": [529.53,  171.14,  313.65,   94.00],
        "di_oligo": [ 431.08,  143.00,    303.52,   79.98],
        "tri_oligo": [1.8,    -1.9,    -6.4,     3.0],
        "deca_oligo":    [0.21,    0.44,   -0.23,   -0.22]
    },

    "Ala": {
        "Nucleotide": ["AMP","CMP","GMP","UMP"],
        "Dia":        [ 0.2146, -1.2585,  0.5038, -0.7893],
        "mo_oligo": [ 564.84,  281.64,  345.67,  130.84]
        "di_oligo": [0.5,     2.4,     1.0,     2.8],
        "tri_oligo": [1.8,    -1.9,    -6.4,     3.0],
        "deca_oligo":    [0.21,    0.44,   -0.23,   -0.22]
    },

    "Met": {
        "Nucleotide": ["AMP","CMP","GMP","UMP"],
        "Dia":        [-3.196, -3.090, -3.356, -2.879],
        "mo_oligo": [ 511.93,  528.71, -123.94,  204.88],
        "di_oligo":  [ 461.52,  523.78,  113.2,   205.76],
        "tri_oligo": [1.8,    -1.9,    -6.4,     3.0],
        "deca_oligo":    [0.21,    0.44,   -0.23,   -0.22]
    }
}


## Update Data and Generate NumPy Array

### Subtask:
Modify the data structures to include 'mono' scale information, filter for specific amino acids and RNA bases, and generate a NumPy array from the processed data.


## Summary:

### Data Analysis Key Findings

*   **'mono' Scale Data Creation**: A new 'mono' scale was generated from the `original_dia_data_for_mono` dataset. This involved mapping amino acid full names to single-letter codes (e.g., Asp to 'D') and nucleotide names to RNA bases (e.g., GMP to 'G'). The resulting 'mono' scale data includes values for amino acids 'D', 'A', 'M' across RNA bases 'G', 'A', 'C', 'U'.
*   **Data Structure**: The `data` dictionary was updated to include the newly generated 'mono' scale, alongside the existing 'mo', 'dimo', 'trimo', and 'decamo' scales. Each scale contains a nested dictionary where keys are RNA bases, and values are dictionaries of amino acid codes to their corresponding numerical data.
*   **Filtering**: The data was filtered to include only specific amino acids (Asparagine 'D', Methionine 'M', Alanine 'A') and RNA bases (Guanine 'G', Adenine 'A', Cytosine 'C', Uracil 'U').
*   **NumPy Array Generation**: A NumPy array named `numpy_array_filtered` was created by iterating through the ordered scales ('mono', 'mo', 'dimo', 'trimo', 'decamo'), then through the filtered amino acids, and finally through the filtered RNA bases.
*   **Array Dimensions**: The final NumPy array has a shape of (15, 4), which corresponds to 5 scales multiplied by 3 amino acids (15 rows) and 4 RNA bases (4 columns).
*   **Missing Values**: In cases where data was not available for a specific scale, RNA base, and amino acid combination, `np.nan` values were inserted into the NumPy array. For instance, in the 'mono' scale, only 'D', 'M', and 'A' amino acids have values, so other amino acid entries within the 'mono' scale for the filtered bases would appear as `np.nan` if they were included in the filtering criteria. The filtered array explicitly extracts only the 'D', 'M', 'A' values.

### Insights or Next Steps

*   The generated NumPy array provides a structured and harmonized dataset suitable for quantitative analysis or machine learning models that require numerical inputs, facilitating cross-scale comparisons for the selected amino acids and RNA bases.
*   Further investigation could involve analyzing the distribution of `np.nan` values across the dataset to understand data completeness and potentially impute or handle these missing values if necessary for subsequent analyses.


<a id="sec5"></a>
## 5. Target-specific array plots

Regenerates per-target summary plots from stored peptide-array scale data, including linear regressions and statistics.

<sub>Source script: <code>peptide_array_target_plots.ipynb</code></sub>

In [ ]:
CHIP_SCALES_PATH = "chip_scales.data"
print(CHIP_SCALES_PATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pickle
import os

# -------------------
# Load chip-scales
# -------------------
CHIP_SCALES_PATH = "chip_scales.data"

with open(CHIP_SCALES_PATH, "rb") as fh:
    chip_scales = pickle.load(fh)

# -------------------
# Your SRSF7 CDS
# -------------------
SRSF7_CDS = (
"ATGTCGCGTTACGGGCGGTACGGAGGAGAAACCAAGGTGTATGTTGGTAACCTGGGAACTGGCGCTGGCA"
"AAGGAGAGTTAGAAAGGGCTTTCAGTTATTATGGTCCTTTAAGAACTGTATGGATTGCGAGAAATCCTCC"
"AGGATTTGCCTTTGTGGAATTCGAAGATCCTAGAGATGCAGAAGATGCAGTACGAGGACTGGATGGAAAG"
"GTGATTTGTGGCTCCCGAGTGAGGGTTGAACTATCGACAGGCATGCCTCGGAGATCACGTTTTGATAGAC"
"CACCTGCCCGACGTCCCTTTGATCCAAATGATAGATGCTATGAGTGTGGCGAAAAGGGACATTATGCTTA"
"T GATTGTCATCGTTACAGCCGGCGAAGAAGAAGCAGGTCACGGTCTAGATCACATTCTCGATCCAGAGGA"
"AGGCGATACTCTCGCTCACGCAGCAGGAGCAGGGGACGAAGGTCAAGGTCAGCATCTCCTCGACGATCAA"
"GATCTATCTCTCTTCGTAGATCAAGATCAGCTTCACTCAGAAGATCTAGGTCTGGTTCTATAAAAGGATC"
"GAGGTATTTCCAATCCCCGTCGAGGTCAAGATCAAGATCCAGGTCTATTTCACGACCAAGAAGCAGCCGA"
"TCAAAGTCCAGATCTCCATCTCCAAAAAGAAGTCGTTCCCCATCAGGAAGTCCTCGCAGAAGTGCAAGTC"
"CTGAAAGAATGGACTGA"
).replace("\n","").replace(" ","")

# -------------------
# DNA → protein translator
# (same simplified logic as original)
# -------------------
CODON_TABLE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGA':'R','AGG':'R','AGT':'S','AGC':'S',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G'
}

def translate_cds(cds):
    cds = cds.upper().replace("U","T")
    aa = []
    for i in range(0, len(cds)-2, 3):
        codon = cds[i:i+3]
        res = CODON_TABLE.get(codon, "X")
        if res == "*":
            break
        aa.append(res)
    return "".join(aa)

# -------------------
# Helper: convert length-L peptide → 10mer key
# -------------------
def pep10(seq):
    L = len(seq)
    if L == 10: return seq
    if L == 1:  return seq*10
    if L == 2:  return seq*5
    if L == 3:  return seq+"G"+seq+"G"+seq
    return seq

# -------------------
# Moving-average (same as original)
# -------------------
def moving_average(a, n=3):
    a = np.array(a, float)
    ret = np.cumsum(a)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n-1:] / n

# -------------------
# Build sequences
# -------------------
RNA_seq = SRSF7_CDS.replace("T","U")   # treat CDS as full cDNA
prot_seq = translate_cds(SRSF7_CDS)
UTR5_length = 0                       # custom sequence: no UTR

print("Protein length:", len(prot_seq))

# -------------------
# Plotting layout from original script
# -------------------
def fig_n_grid(width=12/2.54, grid_spec=(4,2), aspect_ratio=2/3,
               left=0.05, right=0.95, bottom=0.035, top=0.975,
               wspace=0.20, hspace='standard'):
    if hspace=='standard':
        hspace = wspace/aspect_ratio

    panel_w = width*(right-left)*(1 - ((grid_spec[1]-1)/grid_spec[1])*wspace)/grid_spec[1]
    height  = (panel_w*aspect_ratio /
               (1 - ((grid_spec[0]-1)/grid_spec[0])*hspace) *
               grid_spec[0] / (top-bottom))

    fig = plt.figure(figsize=[width, height], dpi=600)
    grid = fig.add_gridspec(grid_spec[0],grid_spec[1],
                            left=left,right=right,
                            bottom=bottom,top=top,
                            wspace=wspace,hspace=hspace)
    return fig, grid

# -------------------
# MAIN: recreate profiles for SRSF7
# -------------------
OUTDIR = "."
os.makedirs(OUTDIR, exist_ok=True)

window = 21*3     # same 21-codon window
RBP = "SRSF7"

for NB in "GACU":

    fig, grid = fig_n_grid()
    axes = [plt.subplot(grid[i,j]) for i in range(4) for j in range(2)]

    i = 0
    for L in [1,2,3,10]:

        ax = axes[i]

        # mRNA nucleotide profile
        cont = [1 if nb == NB else 0 for nb in RNA_seq]
        cont = moving_average(cont, n=window)

        cds_start = UTR5_length
        cds_end   = cds_start + len(prot_seq)*3

        # z-score
        mean_nb = np.mean(cont[cds_start:cds_end])
        std_nb  = np.std(cont[cds_start:cds_end]) or 1
        cont = (np.array(cont)-mean_nb)/std_nb

        # protein affinity profile
        affinities=[]
        for j in range(len(prot_seq)-L+1):
            pep = prot_seq[j:j+L]
            key = pep10(pep)
            if key in chip_scales[NB]:
                val = chip_scales[NB][key]
                affinities += [val,val,val]   # repeat ×3 to match nt scale
            else:
                affinities += [np.nan,np.nan,np.nan]

        affinity_vec = moving_average(affinities, n=window)
        mean_pr = np.nanmean(affinity_vec)
        std_pr  = np.nanstd(affinity_vec) or 1
        affinity_vec = (np.array(affinity_vec)-mean_pr)/std_pr

        # alignment for correlation
        start_corr = UTR5_length + (L//2)
        cont_slice = cont[start_corr:start_corr+len(affinity_vec)]
        min_L = min(len(cont_slice), len(affinity_vec))

        cont_slice = cont_slice[:min_L]
        aff_slice  = affinity_vec[:min_L]

        if len(cont_slice)>=2 and np.all(np.isfinite(cont_slice)) and np.all(np.isfinite(aff_slice)):
            R = stats.pearsonr(cont_slice, aff_slice)[0]
        else:
            R = np.nan

        # plotting
        x = np.arange(len(RNA_seq))

        ax.plot(x[(window//2):(window//2)+len(cont)],
                cont, linewidth=0.5, color="black", alpha=0.6,
                label="mRNA")

        ax.plot(x[start_corr+(window//2): start_corr+(window//2)+len(affinity_vec)],
                affinity_vec,
                color="red", alpha=0.6, linewidth=0.5,
                linestyle=(0,(1,0.4)),
                label=f"protein ({L}-mer)")

        ax.text(0.1,0.95,f"{RBP}, L={L}, R={np.round(R,2)}",
                transform=ax.transAxes,va='top',ha='left')

        ax.set_xlabel("nt | aa×3", labelpad=0.5)
        ax.set_ylabel(f"{NB}-content / affinity (z)", labelpad=0)
        ax.legend()

        i+=1

    out = os.path.join(OUTDIR, f"profile_{RBP}_{NB}.png")
    plt.savefig(out, bbox_inches='tight')
    plt.show()
    plt.close()
    print("saved:", out)


In [ ]:
def fig_n_grid(width=12/2.54, grid_spec=(4, 2), aspect_ratio=2/3,
               left=0.10, right=0.98, bottom=0.10, top=0.95,
               wspace=0.35, hspace=0.45):
    """
    Create a figure and GridSpec with nicer defaults for multi-panel plots.

    width:  total figure width in inches (or cm/2.54 as you used before)
    grid_spec: (n_rows, n_cols)
    aspect_ratio: height/width ratio *per panel* (roughly)
    margins & spacings: matplotlib fraction units
    """

    n_rows, n_cols = grid_spec

    # approximate per-panel size
    panel_width = width * (right - left) / n_cols
    panel_height = panel_width * aspect_ratio

    # total figure height from per-panel height + vertical spacing
    height = panel_height * n_rows * (1 + hspace)

    fig = plt.figure(figsize=(width, height), dpi=300)
    grid = fig.add_gridspec(n_rows, n_cols,
                            left=left, right=right,
                            bottom=bottom, top=top,
                            wspace=wspace, hspace=hspace)
    return fig, grid


In [ ]:
# -------------------
# MAIN: recreate profiles for SRSF7
# -------------------
OUTDIR = "."
os.makedirs(OUTDIR, exist_ok=True)

window = 21*3     # same 21-codon window
RBP = "SRSF7"

for NB in "GACU":

    fig, grid = fig_n_grid()
    axes = [plt.subplot(grid[i,j]) for i in range(4) for j in range(2)]

    i = 0
    for L in [1,2,3,10]:

        ax = axes[i]

        # mRNA nucleotide profile
        cont = [1 if nb == NB else 0 for nb in RNA_seq]
        cont = moving_average(cont, n=window)

        cds_start = UTR5_length
        cds_end   = cds_start + len(prot_seq)*3

        # z-score
        mean_nb = np.mean(cont[cds_start:cds_end])
        std_nb  = np.std(cont[cds_start:cds_end]) or 1
        cont = (np.array(cont)-mean_nb)/std_nb

        # protein affinity profile
        affinities=[]
        for j in range(len(prot_seq)-L+1):
            pep = prot_seq[j:j+L]
            key = pep10(pep)
            if key in chip_scales[NB]:
                val = chip_scales[NB][key]
                affinities += [val,val,val]   # repeat ×3 to match nt scale
            else:
                affinities += [np.nan,np.nan,np.nan]

        affinity_vec = moving_average(affinities, n=window)
        mean_pr = np.nanmean(affinity_vec)
        std_pr  = np.nanstd(affinity_vec) or 1
        affinity_vec = (np.array(affinity_vec)-mean_pr)/std_pr

        # alignment for correlation
        start_corr = UTR5_length + (L//2)
        cont_slice = cont[start_corr:start_corr+len(affinity_vec)]
        min_L = min(len(cont_slice), len(affinity_vec))

        cont_slice = cont_slice[:min_L]
        aff_slice  = affinity_vec[:min_L]

        if len(cont_slice)>=2 and np.all(np.isfinite(cont_slice)) and np.all(np.isfinite(aff_slice)):
            R = stats.pearsonr(cont_slice, aff_slice)[0]
        else:
            R = np.nan

        # plotting
        x = np.arange(len(RNA_seq))

        ax.plot(x[(window//2):(window//2)+len(cont)],
                cont, linewidth=0.5, color="black", alpha=0.6,
                label="mRNA")

        ax.plot(x[start_corr+(window//2): start_corr+(window//2)+len(affinity_vec)],
                affinity_vec,
                color="red", alpha=0.6, linewidth=0.5,
                linestyle=(0,(1,0.4)),
                label=f"protein ({L}-mer)")

        ax.text(0.1,0.95,f"{RBP}, L={L}, R={np.round(R,2)}",
                transform=ax.transAxes,va='top',ha='left')

        ax.set_xlabel("nt | aa×3", labelpad=0.5)
        ax.set_ylabel(f"{NB}-content / affinity (z)", labelpad=0)
        ax.legend()

        i+=1

    out = os.path.join(OUTDIR, f"profile_{RBP}_{NB}.png")
    plt.savefig(out, bbox_inches='tight')
    plt.show()
    plt.close()
    print("saved:", out)


<a id="sec6"></a>
## 6. Homopolymer binding bar plot

Bar plot with error bars comparing mean binding signal of homopolymeric RNA (G10/A10/C10/U10) at a fixed concentration.

<sub>Source script: <code>homopolymer_binding_barplot.ipynb</code></sub>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl


# ---- Data (5 µM RNA) ----
labels = ["G$_{10}$", "A$_{10}$", "C$_{10}$", "U$_{10}$"]

means_5 = np.array([633.375, 804.5, 269.375, 181.625])
rel_err_5 = np.array([9.9, 4.6, 6.5, 6.7])  # %

# Convert to absolute error
err_5 = means_5 * rel_err_5 / 100

# ---- Plot ----
fig, ax = plt.subplots(figsize=(4,3))  # 4:3 ratio

ax.bar(labels, means_5,
       yerr=err_5,
       capsize=5,
       color="grey",
       edgecolor="black",
       linewidth=1)

ax.set_ylabel("Intensity")
ax.set_title("Signal consistency (5 µM RNA)")

plt.tight_layout()
plt.show()

In [ ]:
means_25 = np.array([2755.0, 4940.25, 1591.125, 874.0])
rel_err_25 = np.array([7.4, 5.4, 4.9, 4.2])

err_25 = means_25 * rel_err_25 / 100

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(5,5))  # still 4:3

ax.bar(x - width/2, means_5, width, yerr=err_5,
       capsize=4, label="5 µM",edgecolor="black", color="white")

ax.bar(x + width/2, means_25, width, yerr=err_25,
       capsize=4, label="25 µM",edgecolor="black", color="grey")

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_ylabel("Intensity")
ax.legend()

plt.tight_layout()
plt.show()